# 批量问答：原始 PDF → Markdown → 建库 → 回答问题集 → 整理保存

运行本 Notebook（从上到下依次执行所有 cell）即可完成：

1. 把一批 **原始 PDF** 转成带页码标记（`<!-- page: N -->`）的 Markdown，中间产物落盘、可复用；
2. 用 `superindex.md_ingest.index_markdown` 建库；
3. 读取 **JSON / JSONL 问题集**，规范化成 `superindex.batch.Question`；
4. 用本项目"原服务 agent 链路"（与 `superindex batch` 同一套 client / prefetch / `run_question`）逐题回答；被限流（429 等）时全体暂停 `RATE_LIMIT_WAIT` 秒后重试（最多 `RATE_LIMIT_RETRIES` 次），末尾按出错原因分类统计（出错多时可再单独运行其后的「步骤 4 诊断」cell：只读 `results.jsonl`，并另存 `<运行目录>/diagnosis.txt`）；
5. 整理保存：`summary.md`、`answers.csv`、`answers.xlsx`；

可以在 **Databricks** 上直接作为启动脚本运行，也可以在**本地** Jupyter / VS Code / `nbconvert` 里运行：

| | Databricks | 本地（仓库内） |
|---|---|---|
| 安装 | 运行「环境安装」cell（`%pip install` + 重启 Python） | `uv sync --group notebook`；**不要运行**「环境安装」cell |
| 模型 | 「模型配置」cell 里填 `LLM_CONFIG`（OpenAI 兼容端点填 `OPENAI_MODEL` / `OPENAI_BASE_URL` / `OPENAI_API_KEY`），密钥从 `dbutils.secrets` 读 | `.env`（先读 `ROOT_DIR/.env`，再读当前目录的 `.env`，进程环境变量优先；也可用「模型配置」cell） |
| 数据 | `DATASET_PATH`、`PDF_DIR` 指向 `/Volumes/...` 之类的路径 | 本地路径 |
| 中间产物 | 与本地一致，默认抽取方式 `pdfspine`：用 `<ROOT_DIR>/data/notebook/<题集名>_pdfspine/`（即 `pdfspine_ingest.ipynb` 的产物，直接复用；`PDF_EXTRACTOR=text-layer` / `azure-di` 时为 `<ROOT_DIR>/data/notebook/<题集名>/`）；环境变量 `NB_WORK_DIR` 可改基础目录（pdfspine 时再加 `_pdfspine`），可写进 `.env`；`PERSIST_DIR` 指向持久位置后自动同步 | `<ROOT_DIR>/data/notebook/<题集名>/` |

**在 Databricks 上**：先在 Databricks 上把仓库拉到最新（Repos / Git folder 拉取），再打开本 Notebook 运行；Notebook 直接使用它所在仓库里的代码。

**在 Databricks 上需要填的配置**

1. **安装源**：「环境安装」cell 里默认 `%pip install superindex==0.1.2 ...`（PyPI）；公司内网请换成内部 PyPI 镜像 / Git URL / Repos 路径 / Volumes 里的 wheel（cell 注释里各有一行写法）。
2. **模型配置**：「模型配置」cell 里的 `LLM_CONFIG`（OpenAI 兼容端点 / Databricks Model Serving / Azure OpenAI / 公司 SDK 四种写法；OpenAI 兼容端点优先用 `OPENAI_MODEL` / `OPENAI_BASE_URL` / `OPENAI_API_KEY`）；网关需要自定义请求头时填 `LLM_EXTRA_HEADERS`。
3. **`DATASET_PATH`、`PDF_DIR`**：题集与 PDF 目录，如 `/Volumes/<catalog>/<schema>/<volume>/...`（`DATASET_PATH` 可写在 `.env` 的 `DATASET_PATH=...`；其它配置项同理可用同名环境变量）。`PDF_DIR` 是**递归读取**的目录：子目录里的 `*.pdf` 也会处理。
4. **`PERSIST_DIR`**：如 `/Volumes/<catalog>/<schema>/<volume>/superindex_nb/<题集名>`；不填则不同步（`WORK_DIR` 设在集群本地盘时，集群重启即丢）。

Python 要求 **3.11+**（`superindex` 的 `requires-python`），按 Databricks 官方的 Runtime 与 Python 版本对应关系，对应 Databricks Runtime 15.x 及以上；14.x 自带 Python 3.10，装不上（该对应关系依据官方文档，我们未实测）。

**前置条件**

- 本地：在仓库根目录的 `.env` 里配置问答模型，至少 `SUPERINDEX_CHAT_MODEL`（以及对应的 base url / key，参照 `.env.example`）。**模型必须支持 tool calling。**
  只做步骤 1-3（转换、建库、读题）不需要模型；`INDEX_SUMMARY=True` 时还需要 `SUPERINDEX_INDEX_MODEL`。
- OpenAI 兼容端点也可以只写 `OPENAI_MODEL`（端点上的模型名，自动加 `openai/` 前缀）、`OPENAI_BASE_URL`（一般带 `/v1`）、`OPENAI_API_KEY` 三项（`.env` 或 `LLM_CONFIG` 里都行）：
  **只有 `OPENAI_MODEL` 非空才启用**，单独设了 `OPENAI_API_KEY` / `OPENAI_BASE_URL` 不会改变行为；对应的 `SUPERINDEX_*` 已设时以 `SUPERINDEX_*` 为准（逐变量判断）；
  建库摘要的模型（`INDEX_SUMMARY=True`）在没设 `SUPERINDEX_INDEX_MODEL` 时同用 `OPENAI_MODEL`。「环境」cell 会自己做这组映射，所以 Databricks 上装的 PyPI 版不必支持它。
- 本地依赖：`uv sync --group notebook`（`ipykernel`、`pandas`、`openpyxl`、`pycryptodome` 不在主依赖里）。Databricks 上由「环境安装」cell 安装。
  其中 `pycryptodome` 用于读取 **AES 加密的 PDF**；缺它时这类 PDF 会在步骤 1 标为 `failed` 并提示安装，其它文档照常处理。
- 只改**环境安装 / 模型配置 / 配置**这几个 cell，其余 cell 无需改动；所有 cell 可重复运行（已转换的 PDF、已建好的库、已成功的题都会跳过）。

**带密码的 PDF**（需要密码才能打开的 PDF）

- 主密码写在 `.env` 里的 `PDF_INGEST_PASSWORD`（如 `PDF_INGEST_PASSWORD='你的密码'`，含 `#`、空格等字符时用单引号括起来），它总是第一个被尝试。
- 其它候选密码配 `PDF_PASSWORDS`（配置 cell），排在主密码之后按顺序依次尝试。**不要把明文密码写进 Notebook**：Databricks 上写 `PDF_PASSWORDS = [dbutils.secrets.get(scope="<scope>", key="<key>")]`；
  命令行可用环境变量 `PDF_PASSWORDS`，值是 **JSON 数组字符串**（如 `'["pw1", "pw,with,comma"]'`，密码里可含逗号等任意字符）；值不是 JSON 数组时，整个字符串当作单个密码。
- 不加密、或只设了 owner 密码（打开密码为空）的 PDF 不受影响。需要密码的 PDF 用候选密码依次尝试；密码不对（或没配密码）的**跳过并标记**，不中断其它文档。
- 三种"被跳过"的状态：
  - `skipped_no_text_layer`：PDF 没有文字层（扫描件、纯图片）；已标记则下次不再重复抽取，`FORCE_EXTRACT=True` 才重试；
  - `skipped_wrong_password`：需要密码但没配密码，或候选密码都不对；**每次运行都会重新尝试**（密码配置可能已改对）；
  - `failed`：其它原因抽取失败（如 AES 加密的 PDF 缺 `pycryptodome`），不属于"跳过"。
- 解密只在**内存**里进行（文字层用带密码的 PyPDF2 reader 直接读，补标题用内存里的解密版 PDF），**不会生成解密副本文件**；落盘的只有抽取出的 Markdown 与 `meta.json`（只记 `encrypted` / `password_used` 两个布尔值，不记密码）。
  **注意：Markdown 是明文**，`WORK_DIR` / `PERSIST_DIR` 的访问权限要按原 PDF 的密级管理。
- 需要密码的 PDF 不支持页面截图（`page image`，默认关闭）：步骤 2 不对它们传 `pdf=` 关联。仅 `text-layer` 支持密码；`azure-di` 不处理带密码的 PDF。

**命令行执行（本地）**：必须带上跳过「环境安装」cell 的参数，否则 `%pip install` 会把 PyPI 上的 `superindex` 装进当前环境、覆盖本地的可编辑安装：

```bash
uv run --group notebook jupyter nbconvert --to notebook --execute notebooks/batch_qa.ipynb \
  --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags databricks-setup \
  --output-dir <某目录>
```

（配置项可用同名环境变量覆盖，也可写进 `ROOT_DIR/.env`，见配置 cell。）

**转出的落盘格式**（`pdfspine` / `text-layer` 相同；以下以文字层为例）

- 每个 PDF 存成 `WORK_DIR/md/<PDF 文件名 stem>.md`：UTF-8 Markdown，**每页前有 `<!-- page: N -->` 页码标记**；`ADD_HEADINGS=True` 时在对应页的页码标记后带 `#` 章节标题。
- 旁边的 `<stem>.meta.json` 记录：来源 PDF（`source` 绝对路径、`source_path` 相对 `PDF_DIR` 的路径，建库时由它的文件夹定报告类型）、抽取方式、页数、是否补了标题、空白页（数量与页码）、时间、`status`、`encrypted` / `password_used`（是否加密、是否用了密码，不记密码本身）。
- 这些 Markdown 可以**直接人工查看、修改**；改动后重新运行，步骤 2 会因内容变化自动重新建库（不用 `FORCE_EXTRACT`，那会用 PDF 重新覆盖你的修改）。
- **没有文字层**的 PDF（所有页文字都为空，如扫描件）不写 `.md`、不建库，只写 `meta.json`（`status: "skipped_no_text_layer"`）并登记到 `skipped_pdfs.json`；指向它的题目不会去问模型，而是在结果里记一条 `status=skipped_no_text_layer` 的错误记录。需要处理这类 PDF 请改用 `azure-di`。
- **绝不默认调用 Azure**：即使环境里有 Azure 密钥，也只有显式设 `PDF_EXTRACTOR = "azure-di"` 才会用。
- `PDF_EXTRACTOR="pdfspine"`（**默认**）：用 pdfspine 离线转换（表格转成 Markdown 表格、按字号出 `#` 标题，不用 flash 补标题），落盘格式同上，`meta.json` 的 `extractor` 为 `"pdfspine"`；需要额外安装 `pdfspine==0.12.0`（本地 `uv sync --extra pdfspine`）。可以先用 `notebooks/pdfspine_ingest.ipynb` 转换、建库并与文字层对比，再直接运行本 Notebook：两个 Notebook 用同一个函数 `superindex.extractors.pdfspine.pdfspine_dirs` 推出目录（`PDFSPINE_WORK_DIR` / `PDFSPINE_PERSIST_DIR` 优先，否则 `NB_WORK_DIR` / `PERSIST_DIR` 对应的基础目录名后加 `_pdfspine`），不用手动指定。想用文字层就设 `PDF_EXTRACTOR=text-layer`（目录自动回到不带 `_pdfspine` 的那份）。

**产出物**（所有环境默认都在 `<ROOT_DIR>/data/notebook/<问题集文件名>/` 下，`data/notebook/` 已被 git 忽略；PDF 目录按只读处理，绝不往里写；`ROOT_DIR` 是「环境」cell 打印的绝对路径（仓库根，环境变量 `ROOT_DIR` 可覆盖），相对路径配置都以它为基准；想放集群本地盘就设环境变量 `NB_WORK_DIR`（pdfspine 时它是基础目录，实际目录再加 `_pdfspine`；或直接设 `PDFSPINE_WORK_DIR`）（可写进 `.env`）为本地盘上的绝对路径，用 `PERSIST_DIR` 同步到持久位置）

| 路径 | 内容 |
|---|---|
| `md/` | PDF 转出的 Markdown 及同名 `*.meta.json`，格式见下 |
| `skipped_pdfs.json` | 被跳过的 PDF（无文字层 / 密码不正确或未配置）及原因 |
| `store/` | 索引库 |
| `runs/<时间戳>/results.jsonl` | 逐题完整记录，与 `superindex batch` 格式兼容 |
| `runs/<时间戳>/summary.md`、`answers.csv`、`answers.xlsx` | 总览、答案表（只有三列：`question`、`expected`、`answer`；出错 / 跳过的题 `answer` 为空；csv 为 utf-8-sig，Excel 可直接打开） |


## 环境安装（仅 Databricks）

**只在 Databricks 上运行下面两个 cell**（已打上 cell tag `databricks-setup`）。本地 Jupyter 已经 `uv sync --group notebook` 的，**不要运行**这两个 cell：`%pip install` 会把 PyPI 上的 `superindex` 装进当前环境，覆盖掉本地的可编辑安装。

- 第一个 cell 只放 `%pip install`（Databricks 要求 `%pip` 写在 cell 开头，才是 notebook 作用域安装）。默认从 PyPI 装固定版本 `superindex==0.1.2`；
  集群访问不了公网 PyPI 时，把那一行整行换成下面某一种备选写法（cell 里只保留一行 `%pip`，不要加注释行）。
  `pandas` / `openpyxl` 集群一般已预装，写在命令里只是保证有；本 Notebook 兼容旧版 pandas，不必升级它。
- 第二个 cell 重启 Python，让新装的包生效；只在 Databricks 上才会真的重启，其它环境只打印一句提示。重启会清空内存里的变量，所以它必须在后面所有配置 cell 之前运行。

默认 `PDF_EXTRACTOR="pdfspine"`，第一个 cell 的命令里已含 `pdfspine==0.12.0`；改用 `text-layer` / `azure-di` 时可以去掉它。

依赖由 `%pip` 安装；代码本身用本 Notebook 所在仓库里的版本（「环境」cell 把仓库根插到 `sys.path` 最前面），优先于 `%pip` 装的 PyPI 版。

备选安装写法（任选其一，替换第一个 cell 的内容）：

```
%pip install --index-url https://<公司内部 PyPI 镜像>/simple superindex==0.1.2 pdfspine==0.12.0 pandas openpyxl pycryptodome
%pip install git+https://<git 服务器>/<组织>/SuperIndex.git@<tag 或 commit> pdfspine==0.12.0 pandas openpyxl pycryptodome
%pip install /Workspace/Repos/<用户>/SuperIndex pdfspine==0.12.0 pandas openpyxl pycryptodome
%pip install /Volumes/<catalog>/<schema>/<volume>/wheels/superindex-0.1.2-py3-none-any.whl pdfspine==0.12.0 pandas openpyxl pycryptodome
```

命令行用 `nbconvert` 在本地执行时，用下面的参数跳过这两个 cell：

```bash
uv run --group notebook jupyter nbconvert --to notebook --execute notebooks/batch_qa.ipynb \
  --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags databricks-setup \
  --output-dir <某目录>
```

In [ ]:
%pip install superindex==0.1.2 pdfspine==0.12.0 pandas openpyxl pycryptodome

In [ ]:
import os

# 只在 Databricks 上重启 Python（让上一格 %pip 装的包生效）；其它环境什么都不做
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()  # noqa: F821 - dbutils 由 Databricks 注入
else:
    print("非 Databricks 环境，跳过")

In [ ]:
# ───────────── 模型配置（必须在任何 import superindex / litellm 之前运行）─────────────
# 本 cell 只写环境变量，不 import litellm / superindex，所以下面这些变量能赶在它们之前生效。
# 默认不改动任何环境变量：本地有 .env 时行为不变（.env 在下一格"环境"里才加载，且不覆盖这里已设的变量）。
import os

os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")  # litellm 用内置价格表，import 时不联网拉取
os.environ.setdefault("OPENAI_AGENTS_DISABLE_TRACING", "1")    # 关闭 openai-agents 的 tracing 上报

# 想改哪项就取消注释并填值；值为空串的项会被跳过。**不要把密钥明文写进 Notebook**。
# 在 Databricks 上从 secrets 读：dbutils.secrets.get(scope="<scope>", key="<key>")
LLM_CONFIG: dict[str, str] = {
    # ── 方式一：OpenAI 兼容端点（vLLM、公司网关等）：端点上的模型名（自动加 openai/ 前缀），地址一般带 /v1 ──
    # 只有 OPENAI_MODEL 非空才启用这一组；对应的 SUPERINDEX_* 已设（非空）时以 SUPERINDEX_* 为准。映射在下一格"环境"里做。
    # "OPENAI_MODEL": "<模型名>",
    # "OPENAI_BASE_URL": "https://<公司网关>/v1",
    # "OPENAI_API_KEY": dbutils.secrets.get(scope="<scope>", key="<key>"),
    # 等价的 SUPERINDEX_* 写法（也可以用）：
    # "SUPERINDEX_CHAT_MODEL": "openai/<模型名>",
    # "SUPERINDEX_BASE_URL": "https://<公司网关>/v1",
    # "SUPERINDEX_API_KEY_OVERRIDE": dbutils.secrets.get(scope="<scope>", key="<key>"),
    #
    # ── 方式二：Databricks Model Serving：databricks/<serving endpoint 名> ──
    # "SUPERINDEX_CHAT_MODEL": "databricks/<endpoint 名>",
    # "SUPERINDEX_BASE_URL": "https://<workspace 域名>/serving-endpoints",
    # "SUPERINDEX_API_KEY_OVERRIDE": dbutils.secrets.get(scope="<scope>", key="<key>"),  # Databricks token
    #
    # ── 方式三：Azure OpenAI：azure/<部署名> ──
    # "SUPERINDEX_CHAT_MODEL": "azure/<deployment 名>",
    # "AZURE_API_BASE": "https://<资源名>.openai.azure.com",
    # "AZURE_API_KEY": dbutils.secrets.get(scope="<scope>", key="<key>"),
    # "AZURE_API_VERSION": "<api 版本>",
    #
    # ── 方式四：公司自封装的 Python SDK：模型名写 company/<任意名>，并启用下面的"公司 SDK 适配器"cell ──
    # "SUPERINDEX_CHAT_MODEL": "company/<任意名>",
    #
    # ── 其它 ──
    # "SUPERINDEX_INDEX_MODEL": "",         # 只在 INDEX_SUMMARY=True 时用到；写法同 SUPERINDEX_CHAT_MODEL；用 OPENAI_MODEL 时不设则同用它
    # "SUPERINDEX_REASONING_EFFORT": "",    # 公司模型一般留空；设了而模型不支持会报 UnsupportedParamsError
}

# 公司网关要求的自定义请求头（如 {"X-Gateway-Token": dbutils.secrets.get(...)}）。默认空 = 不加。
# 只作用于问答（步骤 4）；建库摘要（INDEX_SUMMARY=True）这条路径不带这些 header。
LLM_EXTRA_HEADERS: dict[str, str] = {}

for _name, _value in LLM_CONFIG.items():
    if _value:
        os.environ[_name] = _value
print("模型配置：写入环境变量 " + (", ".join(k for k, v in LLM_CONFIG.items() if v) or "（无，沿用 .env / 进程环境变量）"))
print("自定义请求头：" + (", ".join(LLM_EXTRA_HEADERS) or "（无）"))

In [ ]:
# 确定唯一的绝对路径基准 ROOT_DIR（.env、data/、相对路径配置都以它为基准，不依赖 cwd、不 chdir）。必须在 import superindex 之前做。
# ROOT_DIR：环境变量 ROOT_DIR > 从 cwd 向上找到的仓库根。本 Notebook 必须在仓库里运行（Databricks 上先把仓库拉到最新再打开）。
import io
import json
import os
import re
import shutil
import statistics
import sys
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from typing import Any

from dotenv import dotenv_values, load_dotenv
import pandas as pd
from IPython.display import display

try:
    from tqdm import tqdm
except ImportError:  # 没有 tqdm 就退化为 print
    tqdm = None


def find_repo_root(start: Path) -> Path | None:
    """向上找含 pyproject.toml 和 superindex/ 包目录的目录（即本仓库根）；找不到（pip 安装、不在仓库里）返回 None。"""
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file() and (folder / "superindex").is_dir():
            return folder
    return None


IS_DATABRICKS = bool(os.environ.get("DATABRICKS_RUNTIME_VERSION"))
_repo_root = find_repo_root(Path.cwd().resolve())
if _repo_root is None:
    raise RuntimeError(
        f"从当前目录 {Path.cwd().resolve()} 向上找不到 SuperIndex 仓库根（含 pyproject.toml 和 superindex/）。"
        "请在仓库里打开并运行本 Notebook（Databricks 上先把仓库拉到最新）。"
    )
ROOT_DIR = Path(os.environ.get("ROOT_DIR", "").strip() or _repo_root).expanduser().resolve()
sys.path.insert(0, str(_repo_root))  # import superindex 用仓库里的源码，而不是 %pip 装的 PyPI 版；不 chdir


def under_root(p: str | Path) -> Path:
    """绝对路径原样（resolve）；相对路径以 ROOT_DIR 为基准。"""
    p = Path(p).expanduser()
    return (p if p.is_absolute() else ROOT_DIR / p).resolve()


_root_from = "环境变量 ROOT_DIR" if os.environ.get("ROOT_DIR", "").strip() else "从当前目录向上找到的仓库根"
print(f"当前目录 : {Path.cwd().resolve()}")
print(f"ROOT_DIR : {ROOT_DIR}（来源：{_root_from}）")

from superindex.runtime import (
    ConfigError,
    LLMSettings,
    configure_litellm,
    getenv,
    resolve_instructions,
    set_offline_defaults,
)

set_offline_defaults()  # 必须早于任何 litellm 的 import
# .env 候选位置：先 ROOT_DIR/.env，再当前目录的 .env（与前者不是同一个文件时）。都 override=False，
# 所以优先级：进程环境变量 > ROOT_DIR/.env > 当前目录/.env。
_env_candidates = [ROOT_DIR / ".env"]
if (Path.cwd().resolve() / ".env").resolve() != _env_candidates[0].resolve():
    _env_candidates.append(Path.cwd().resolve() / ".env")
ENV_SOURCE: dict[str, str] = {k: "环境变量" for k in os.environ}  # 变量名 -> 来源（只记名字，不记值）
ENV_FILES: list[Path] = []
_dotenv_keys: set[str] = set()
for _env_file in _env_candidates:
    if not _env_file.is_file():
        print(f".env     : {_env_file} 不存在")
        continue
    _keys = sorted(k for k, v in dotenv_values(_env_file, encoding="utf-8").items() if v is not None)
    _labels = []
    for _k in _keys:
        _shadow = ENV_SOURCE.get(_k)  # 已有来源 = 环境变量，或前一个 .env
        ENV_SOURCE.setdefault(_k, f".env（{_env_file}）")
        _labels.append(_k if _shadow is None else _k + ("（已被环境变量覆盖）" if _shadow == "环境变量" else "（已被前一个 .env 覆盖）"))
    _dotenv_keys.update(_keys)
    ENV_FILES.append(_env_file)
    load_dotenv(_env_file, override=False, encoding="utf-8")  # 已设置的环境变量优先，不覆盖
    print(f".env     : {_env_file} 已加载，{len(_keys)} 个变量：{', '.join(_labels) or '（空）'}")  # 只打印变量名，绝不打印值
if not ENV_FILES:
    print(f"未读到任何 .env：配置项将使用环境变量或默认值。请把 .env 放到 {ROOT_DIR} 或 {Path.cwd().resolve()}")
_old_names = sorted(k for k in {*os.environ, *_dotenv_keys} if k.startswith("SI_NB_"))
if _old_names:
    _renamed = {"SI_NB_PDF_DIR": "PDF_DIR", "SI_NB_WORK_DIR": "NB_WORK_DIR", "SI_NB_DATASET": "DATASET_PATH",
                "SI_NB_RESULTS": "RESULTS_PATH", "SI_NB_EVAL_INPUT": "RESULTS_PATH"}
    print("⚠ 发现已废弃的变量名（不再生效）："
          + "；".join(f"{k} → " + ("请改为 " if i == 0 else "") + _renamed.get(k, k[len("SI_NB_"):])
                      for i, k in enumerate(_old_names)))


def apply_openai_env() -> dict[str, str]:
    """OPENAI_MODEL / OPENAI_BASE_URL / OPENAI_API_KEY -> SUPERINDEX_*（与新版 superindex.runtime 同一规则；
    这里自己做一遍，PyPI 旧版没有该回退时行为也一致）。只有 OPENAI_MODEL 非空才启用；
    SUPERINDEX_*（含旧名 PAGEINDEX_*）已设为非空的不覆盖。返回实际写入的 {变量名: 值}。"""
    model = os.environ.get("OPENAI_MODEL", "").strip()
    if not model:
        return {}
    model = model if model.startswith("openai/") else f"openai/{model}"  # 模型名含 "/" 也要加，别让 litellm 把前半当 provider
    wanted = {
        "SUPERINDEX_CHAT_MODEL": model,
        "SUPERINDEX_INDEX_MODEL": model,  # 建库摘要（INDEX_SUMMARY=True）同用这个模型
        "SUPERINDEX_BASE_URL": os.environ.get("OPENAI_BASE_URL", "").strip(),
        "SUPERINDEX_API_KEY_OVERRIDE": os.environ.get("OPENAI_API_KEY", "").strip(),
    }
    applied = {}
    for name, value in wanted.items():
        if value and not (getenv(name) or "").strip():
            os.environ[name] = applied[name] = value
    return applied


_applied = apply_openai_env()
if os.environ.get("OPENAI_MODEL", "").strip():
    print("模型配置：使用 OPENAI_MODEL / OPENAI_BASE_URL / OPENAI_API_KEY"
          f"（模型 {getenv('SUPERINDEX_CHAT_MODEL')}，地址 {getenv('SUPERINDEX_BASE_URL') or '（未设）'}；"
          f"已填入 {', '.join(_applied) or '（无，SUPERINDEX_* 均已设置）'}）")  # 不打印 key

from superindex import page_images, prefetch
from superindex.batch import (
    RESULTS_FILE,
    SUMMARY_FILE,
    Question,
    _from_item,
    _routing_policy,
    _scope,
    judge_hits,
    read_results,
    route_diagnostics,
    route_scope,
    route_summary_lines,
    calc_summary_lines,
    run_question,
    score,
    write_summary,
)
from superindex.cli import make_client
from superindex.engine.local_store import DocStore
from superindex.extractors.backend import Extractor
from superindex.md_ingest import (PAGE_MARKER_RE, RateLimitPolicy, backfill_doc_meta, doc_meta_stats,
                                  clash_names, drop_clash_leftovers, index_markdown, pdf_source_path,
                                  same_name_sources)

print(f"运行环境: {'Databricks' if IS_DATABRICKS else '本地'}")
print(f"仓库根目录: {_repo_root}")


def _superindex_version(src: Path) -> str:
    try:
        if (src.parent / "pyproject.toml").is_file():  # 源码树：读 pyproject 里的版本（已安装包的元数据可能是别的副本的）
            import tomllib

            return tomllib.loads((src.parent / "pyproject.toml").read_text(encoding="utf-8"))["project"]["version"]
        from importlib.metadata import version

        return version("superindex")
    except Exception:  # noqa: BLE001
        return "未知"


import superindex

_SRC = Path(superindex.__file__).resolve().parent
print(f"superindex 来自: {_SRC}（版本 {_superindex_version(_SRC)}）")

In [ ]:
# ───────────── 配置（所有可调项都在这一格）─────────────
# 每项都可用同名环境变量覆盖（方便命令行 nbconvert 执行；也可写进 ROOT_DIR/.env），例如 LIMIT=3。
# 例外：工作目录 WORK_DIR 的环境变量叫 NB_WORK_DIR（见下）。
def _env(name: str, default: Any, cast=str) -> Any:
    raw = os.environ.get(name)
    return default if raw is None or raw.strip() == "" else cast(raw.strip())


def _source(name: str) -> str:
    """配置项的值来自哪里：环境变量非空 → 查 ENV_SOURCE（环境变量 / .env（路径）），否则默认值。"""
    return ENV_SOURCE.get(name, "环境变量") if os.environ.get(name, "").strip() else "默认值"


def _raw_note(name: str, default: str) -> str:
    """原始写法是相对路径时，附带显示原始值（相对路径以 ROOT_DIR 为基准）。"""
    raw = os.environ.get(name, "").strip() or default
    return "" if Path(raw).expanduser().is_absolute() else f"（原始值 {raw}，以 ROOT_DIR 为基准）"


def _bool(text: str) -> bool:
    return text.lower() in ("1", "true", "yes", "on")


# --- 输入 ---
# Databricks 上写 Volumes / DBFS 的 FUSE 路径，如 "/Volumes/<catalog>/<schema>/<volume>/qa/questions.jsonl"、
# "/dbfs/<目录>/questions.jsonl"（占位符请换成实际值）。
# 问题文件路径优先读环境变量 DATASET_PATH（写进 .env 即可：DATASET_PATH=data/golden_questions.json；上一格已加载 .env），
# 没设用默认值。相对路径以 ROOT_DIR 为基准（下面所有路径配置同理）。
DATASET_PATH = under_root(_env("DATASET_PATH", "data/questions.jsonl"))  # 问题文件：.json 或 .jsonl
# 如 "/Volumes/<catalog>/<schema>/<volume>/qa/pdfs"、"/dbfs/<目录>/pdfs"
PDF_DIR = under_root(_env("PDF_DIR", "data/pdfs"))                          # 原始 PDF 目录（递归查找 *.pdf）
# 诊断打印（带来源和存在性）：放在各项检查之前，检查抛错时关键路径也已经打印出来
print(f"问题集   : {DATASET_PATH}   [来源：{_source('DATASET_PATH')}]   [{'存在' if DATASET_PATH.is_file() else '不存在'}]"
      + _raw_note("DATASET_PATH", "data/questions.jsonl"))
_pdf_state = (f"存在，递归找到 {sum(1 for p in PDF_DIR.rglob('*') if p.is_file() and p.suffix.lower() == '.pdf')} 个 *.pdf"
              if PDF_DIR.is_dir() else "不存在")
print(f"PDF 目录 : {PDF_DIR}   [来源：{_source('PDF_DIR')}]   [{_pdf_state}]" + _raw_note("PDF_DIR", "data/pdfs"))

# 数据集字段名映射：{规范名: 数据集里的字段名或路径}。
# expected / doc / id / pages 可缺省：值设为 None，或数据里没有该字段（含路径取不到）都行。
# doc 缺省 = 在全库范围提问。其余字段（包括被映射走的嵌套源字段，如整个 evidence）原样带进结果（Question.extra）。
#
# 路径语法（值除了顶层字段名，还可以是路径；顶层字段名本身含 "." 的极端情况不支持）：
#   "meta.source"      用 "." 逐层取嵌套对象
#   "evidence[].file"  "[]" 展开列表：取每个元素的 file，得到列表；去重、保持首次出现顺序、丢弃空值
# 取到列表时：id / question / expected 只有 1 个元素就取该元素，多个用换行拼成字符串；
#            doc 列表即多个文档（字符串沿用原行为）；
#            pages（可选，默认 None）：检索命中判定用的页码（batch 的 `pages` 字段，元素可为 12 或 "12-13"）。
# 示例（题目带 evidence 列表、每项有 file / page 的数据集）：
#   {"id": "id", "question": "question", "expected": "ground_truth",
#    "doc": "evidence[].file", "pages": "evidence[].page"}
# 注意：把 doc 映射到标准证据文档，会把检索范围限定在正确文档内，评测成绩会偏乐观；
#       想评估"全库检索"就把 doc 设为 None。pages 多文档时是各文档页码的并集（batch 只按页码判定，不区分文档）。
# 可用环境变量 FIELD_MAP 传入整个映射（JSON 字符串）。
FIELD_MAP = _env("FIELD_MAP", {"id": "id", "question": "question", "expected": "expected", "doc": "doc",
                               "pages": None}, json.loads)
# JSON 顶层是 {"questions": [...]} 或直接列表时无需设置；顶层是其它键名（如 {"items": [...]}）时填这个键名。
RECORDS_KEY: str | None = _env("RECORDS_KEY", None)

# --- PDF → Markdown ---
PDF_EXTRACTOR = _env("PDF_EXTRACTOR", "pdfspine")    # "pdfspine"（默认）：pdfspine 离线转换，保留表格（需安装 pdfspine）；直接使用 notebooks/pdfspine_ingest.ipynb 生成的 md/store
                                                     # "text-layer"：PDF 文字层（原始 PageIndex 做法，完全离线）；
                                                     # "azure-di"：Azure Document Intelligence（需 AZURE_DI_ENDPOINT / AZURE_DI_KEY）
ADD_HEADINGS = _env("ADD_HEADINGS", True, _bool)     # 仅 text-layer：用离线版面分析（flash）给 Markdown 补 # 章节标题
FORCE_EXTRACT = _env("FORCE_EXTRACT", False, _bool)  # True：已有 .md 也重新转换（改了 PDF_EXTRACTOR / ADD_HEADINGS 后需要）

# --- 带密码的 PDF（需要密码才能打开的 PDF；默认不处理）---
# 主密码放在环境变量 PDF_INGEST_PASSWORD 里（写进 .env 即可：PDF_INGEST_PASSWORD='你的密码'；上一格已加载 .env）。
# 它原样当作一个密码（不做 JSON 解析），总是排在候选列表第一个。.env 里密码含 #、空格、引号等字符时用单引号括起来。
# 其它候选密码填 PDF_PASSWORDS，排在主密码之后依次尝试。都没配 = 需要密码的 PDF 一律跳过并标记（不影响其它 PDF）。
# 不加密、或只设了 owner 密码（打开密码为空）的 PDF 不受影响，照常处理。
# 不要把明文密码写进 Notebook。Databricks 上没有 .env 时从 secrets 读（把下一行 [] 换成）：
#   PDF_PASSWORDS = [dbutils.secrets.get(scope="<scope>", key="<key>")]  # 多个密码就写多项
# 也可用环境变量 PDF_PASSWORDS 覆盖：值是 JSON 数组字符串，如 '["pw1", "pw,with,comma"]'（密码里可含逗号等任意字符，
# 与 FIELD_MAP 的 JSON 传法一致）；值不是 JSON 数组（不是合法 JSON，或是 JSON 数字/字符串）时，整个字符串当作单个密码。
# 密码不会被写进 meta.json / skipped_pdfs.json / 任何输出；步骤 1 只在内存里解密，不落解密副本。
PDF_PASSWORDS: list[str] = []


def parse_passwords(raw: str) -> list[str]:
    """环境变量值 -> 候选密码列表：JSON 数组按元素；其它（含非法 JSON）整个字符串当单个密码。"""
    try:
        data = json.loads(raw)
    except ValueError:
        return [raw]
    return [str(p) for p in data if p not in (None, "")] if isinstance(data, list) else [raw]


_env_passwords = os.environ.get("PDF_PASSWORDS")
if _env_passwords is not None and _env_passwords.strip():
    PDF_PASSWORDS = parse_passwords(_env_passwords)
_main_password = os.environ.get("PDF_INGEST_PASSWORD", "")
PDF_PASSWORDS = [_main_password, *PDF_PASSWORDS]  # 主密码排第一
PDF_PASSWORDS = list(dict.fromkeys(p for p in PDF_PASSWORDS if p))  # 去重保序；空串不是密码（空密码本来就会先试）

# --- 中间产物目录 ---
# text-layer / azure-di：所有环境默认 ROOT_DIR/data/notebook/<题集名>（所有运行产物都放 ROOT_DIR/data/ 下）。
# pdfspine（默认）：目录由下面的"文字层基础目录"推出，规则与 pdfspine_ingest 完全相同（共用 pdfspine_dirs），所以直接用它的产物：
# 工作目录 = PDFSPINE_WORK_DIR，没设则基础目录名后加 _pdfspine；持久目录 = PDFSPINE_PERSIST_DIR，没设则 PERSIST_DIR 名后加 _pdfspine。
# 此时 NB_WORK_DIR / PERSIST_DIR 是"基础目录"（与 pdfspine_ingest 里同名变量的含义一致），不是 pdfspine 的实际目录。Databricks 上想放集群本地盘（FUSE 路径不一定支持存储层用的
# flock / 原子替换 / 逐行追加），设环境变量 NB_WORK_DIR 为本地盘（local_disk0）上的绝对路径即可（写进 .env 即可：NB_WORK_DIR=/local_disk0/...；上一格已加载 .env）。
TEXT_WORK_DIR = under_root(os.environ.get("NB_WORK_DIR", "").strip() or ROOT_DIR / "data" / "notebook" / DATASET_PATH.stem)
_text_persist = _env("PERSIST_DIR", None, Path)
TEXT_PERSIST_DIR: Path | None = under_root(_text_persist) if _text_persist else None
WORK_DIR, PERSIST_DIR = TEXT_WORK_DIR, TEXT_PERSIST_DIR
if PDF_EXTRACTOR == "pdfspine":
    from superindex.extractors.pdfspine import pdfspine_dirs
    WORK_DIR, PERSIST_DIR = pdfspine_dirs(TEXT_WORK_DIR, TEXT_PERSIST_DIR, root=ROOT_DIR)
_pdfspine_work = PDF_EXTRACTOR == "pdfspine" and bool(os.environ.get("PDFSPINE_WORK_DIR", "").strip())
_work_name = "PDFSPINE_WORK_DIR" if _pdfspine_work else "NB_WORK_DIR"
_work_src = (f"{_work_name}（{_source(_work_name)}）" if _source(_work_name) != "默认值" else "默认值") + (
    "，基础目录名后加 _pdfspine" if PDF_EXTRACTOR == "pdfspine" and not _pdfspine_work else "")
print(f"工作目录 : {WORK_DIR}   [来源：{_work_src}]"
      f"   [{'已存在' if WORK_DIR.is_dir() else '尚未创建（首次写入时自动创建）'}]" + _raw_note(_work_name, str(WORK_DIR)))
# PDF 目录按只读处理：工作目录（md/ store/ runs/）绝不能落在里面（也避免把产物当成输入 PDF 递归扫到）
if WORK_DIR.resolve().is_relative_to(PDF_DIR.resolve()):
    raise ValueError(f"工作目录落在 PDF 目录之内；PDF 目录按只读处理，请修改 NB_WORK_DIR 或 PDF_DIR。WORK_DIR={WORK_DIR}，PDF_DIR={PDF_DIR}")
# 持久目录（默认 None = 不同步）。Databricks 上建议填持久路径，如 "/Volumes/<catalog>/<schema>/<volume>/superindex_nb/<题集名>"：
# 开始时把里面已有的 md/ store/ runs/ skipped_pdfs.json 恢复到本地 WORK_DIR（集群重启后不必重抽取、重建库，已答的题可续跑），
# 步骤 2 建库后和全部完成后再把 WORK_DIR 复制回去。
_pdfspine_persist = PDF_EXTRACTOR == "pdfspine" and bool(os.environ.get("PDFSPINE_PERSIST_DIR", "").strip())
_persist_name = "PDFSPINE_PERSIST_DIR" if _pdfspine_persist else "PERSIST_DIR"
_persist_src = f"{_persist_name}（{_source(_persist_name)}）" + ("，由 PERSIST_DIR 名后加 _pdfspine 推出" if PDF_EXTRACTOR == "pdfspine" and not _pdfspine_persist else "")
print(f"持久目录 : {PERSIST_DIR}   [来源：{_persist_src}]   [{'已存在' if PERSIST_DIR.is_dir() else '尚不存在'}]"
      + _raw_note(_persist_name, str(PERSIST_DIR)) if PERSIST_DIR else "持久目录 : （未设置，不同步）")
if IS_DATABRICKS and PERSIST_DIR is None and not str(WORK_DIR).startswith(("/Volumes/", "/dbfs/", "/Workspace/")):
    print(f"⚠⚠ 注意：WORK_DIR 在集群本地盘（{WORK_DIR}），且未设置 PERSIST_DIR —— 集群重启后 md/ store/ runs/ 全部丢失，"
          "需要重新抽取、建库（重新调 LLM）。建议设置 PERSIST_DIR 为 /Volumes/... 下的持久路径")
if PERSIST_DIR is not None and PERSIST_DIR.resolve().is_relative_to(PDF_DIR.resolve()):
    raise ValueError(f"持久目录落在 PDF 目录之内；PDF 目录按只读处理，请修改 PERSIST_DIR 或 PDF_DIR。PERSIST_DIR={PERSIST_DIR}，PDF_DIR={PDF_DIR}")
MD_DIR = WORK_DIR / "md"        # 转出的 Markdown
STORE = WORK_DIR / "store"      # 索引库
RUNS_DIR = WORK_DIR / "runs"    # 每次运行一个时间戳目录

# --- 建库 ---
INDEX_SUMMARY = _env("INDEX_SUMMARY", False, _bool)  # True：建库时用 LLM（SUPERINDEX_INDEX_MODEL）写节点摘要；False：不调 LLM
# 文档元数据（与 INDEX_SUMMARY 无关）：每份文档一次 LLM 调用，抽公司 / 地区 / 主报告期 / 报告类型 / 一句话描述，存进 doc.json
# 的 metadata.doc_meta（description 为空时也填上）。模型用 SUPERINDEX_INDEX_MODEL，没设用 SUPERINDEX_CHAT_MODEL；都没配或调用失败时
# 只用文件名规则（如 AIA_AR2024 → FY2024 年报）。已建好的库重跑时自动补齐缺的元数据，不重建树。
DOC_META = _env("DOC_META", True, _bool)
# 建库时（元数据 / 摘要）模型接口额度不足或被限流：等多久（秒）后重试同一文档、每个文档最多等几次；0 = 不等（直接退回）。
# 某个文档等满仍被限流 = 判断为额度耗尽：本次运行后续文档不再等待（直接按失败处理，下次运行重试）。
# 等满仍失败：元数据退回文件名规则、不记为已完成（下次运行重新抽取）；摘要失败该文档记"失败"、不入库（下次运行重建）。
INDEX_RATE_LIMIT_WAIT = _env("INDEX_RATE_LIMIT_WAIT", 60.0, float)
INDEX_RATE_LIMIT_MAX_WAITS = _env("INDEX_RATE_LIMIT_MAX_WAITS", 30, int)

# --- 期间路由（题目没写 doc 时生效；写了 doc 的题保持原行为）---
ROUTE = _env("ROUTE", True, _bool)                    # 按问题里的年份 / 期间限定检索范围；问题无期间或零命中则退回全库
ROUTE_ADJACENT = _env("ROUTE_ADJACENT", True, _bool)  # 把下一年的报告也纳入（其中有本年的对比数）

# --- 问答 ---
LIMIT: int | None = _env("LIMIT", None, int)    # 只跑前 N 题；None = 全部
TIMEOUT = _env("TIMEOUT", 600.0, float)         # 每题超时（秒）；回合上限放宽后单题更慢，别设太小
def _turns(text: str) -> int | None:
    """MAX_TURNS 的取值：正整数；≤0 或 none / default = 不传（沿用 openai-agents SDK 默认的 10 回合）。"""
    return None if text.lower() in ("none", "default") or int(text) <= 0 else int(text)
MAX_TURNS: int | None = _env("MAX_TURNS", 25, _turns)  # 每题 agent 最多几回合；超过记为 max_turns 出错。None = 沿用 SDK 默认（10）
CONCURRENCY = _env("CONCURRENCY", 1, int)       # 并发题数
RESUME = _env("RESUME", True, _bool)            # True：续跑最近一次运行目录（跳过已成功的题、重跑出错的题）；False：新开时间戳目录
PREFETCH: bool | None = _env("PREFETCH", None, _bool)  # 关键词检索前置：None = 沿用项目默认（SUPERINDEX_PREFETCH，默认开）
PREFETCH_K: int | None = _env("PREFETCH_K", None, int)  # 前置页数：None = 沿用项目默认（5）
RATE_LIMIT_WAIT = _env("RATE_LIMIT_WAIT", 60.0, float)  # 模型接口被限流（429 等）时所有线程一起暂停多久（秒）再重试
RATE_LIMIT_RETRIES = _env("RATE_LIMIT_RETRIES", 5, int)  # 每题被限流后最多重试几次；0 = 关闭（限流直接记为出错）

print(f"ROOT_DIR : {ROOT_DIR}")
print(f"MD_DIR   : {MD_DIR}")
print(f"STORE    : {STORE}")
print(f"RUNS_DIR : {RUNS_DIR}")
print(f"抽取方式 : {PDF_EXTRACTOR}（补标题：{ADD_HEADINGS if PDF_EXTRACTOR == 'text-layer' else '不适用'}）   [来源：{_source('PDF_EXTRACTOR')}]"
      + ("；切回文字层：设 PDF_EXTRACTOR=text-layer" if PDF_EXTRACTOR != "text-layer" else ""))
print(f"PDF 密码 : 已配置 {len(PDF_PASSWORDS)} 个候选密码")  # 只打印个数，绝不打印密码本身


# --- 出错原因分类（步骤 4 与「步骤 4 诊断」cell 共用的纯函数，不依赖模型 / 索引库）---
# 限流判定（与步骤 2 建库共用 superindex.engine.errors.is_rate_limit_error）：429 只认 "code / status / HTTP 429"、
# "429 Too Many Requests" 这类写法，避免页码等数字误判
from superindex.engine.errors import is_rate_limit_error

ERROR_KINDS = {"max_turns": "回合上限(max_turns)", "rate_limit": "限流", "timeout": "超时",
               "backend": "模型接口其它错误", "other": "其它"}


def is_rate_limited(error: str | None) -> bool:
    """record["error"] 是不是模型接口限流（429 / rate limit / quota 等）；回合上限、超时不算。"""
    if not error or "max_turns" in error or error.startswith("timeout after"):
        return False
    return is_rate_limit_error(error)


def error_kind(error: str) -> str:
    """出错记录的原因分类，键见 ERROR_KINDS。"""
    if "max_turns" in error:
        return "max_turns"
    if is_rate_limited(error):
        return "rate_limit"
    if error.startswith("timeout after"):
        return "timeout"
    if "model backend failed" in error or "agent backend failed" in error:
        return "backend"
    return "other"


In [ ]:
# ───────────── 持久化目录同步（仅当配置了 PERSIST_DIR；否则下面的函数什么都不做）─────────────
PERSIST_ITEMS = ("md", "store", "runs", "skipped_pdfs.json")  # 开始时从 PERSIST_DIR 恢复的内容


def _copy_file(src: Path, dst: Path) -> bool:
    """复制一个文件；dst 已有同样大小、且修改时间不早于 src（容差 1s，FUSE 时间戳精度低）的视为未变、跳过。
    返回是否复制了。只复制内容、不复制权限（FUSE 路径上 copystat 常失败）；修改时间尽量设成与 src 一致，失败忽略。"""
    st = src.stat()
    try:
        dst_st = dst.stat()
        if dst_st.st_size == st.st_size and dst_st.st_mtime >= st.st_mtime - 1:
            return False
    except OSError:
        pass
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src, dst)
    try:
        os.utime(dst, (st.st_atime, st.st_mtime))
    except OSError:
        pass
    return True


def _copy_files(src: Path, dst: Path) -> tuple[int, int]:
    """把 src（文件或目录）复制到 dst，只复制有变化的文件；返回 (复制的文件数, 总文件数)。"""
    if src.is_file():
        return int(_copy_file(src, dst)), 1
    files = [p for p in src.rglob("*") if p.is_file()]
    return sum(_copy_file(p, dst / p.relative_to(src)) for p in files), len(files)


def _stale_docs(persist: Path) -> list[str]:
    """PERSIST_DIR/store/docs 里本地 store 已不存在的文档目录名（重建库时旧 doc_id 被删）；只列出，不删除。"""
    local, remote = WORK_DIR / "store" / "docs", persist / "store" / "docs"
    if not local.is_dir() or not remote.is_dir():
        return []
    return sorted(d.name for d in remote.iterdir() if d.is_dir() and not (local / d.name).exists())


def _persist_dir() -> Path | None:
    if PERSIST_DIR is None:
        return None
    if PERSIST_DIR.resolve() == WORK_DIR.resolve():
        print("PERSIST_DIR 与 WORK_DIR 相同，无需同步")
        return None
    return PERSIST_DIR


def restore_from_persist() -> None:
    """WORK_DIR 里还没有的 md/store/runs/skipped_pdfs.json，从 PERSIST_DIR 复制进来（集群重启后接着上次的进度）。"""
    persist = _persist_dir()
    if persist is None:
        return
    for name in PERSIST_ITEMS:
        src, dst = persist / name, WORK_DIR / name
        if not src.exists() or dst.exists():
            continue
        try:
            print(f"从 PERSIST_DIR 恢复 {name}（{_copy_files(src, dst)[1]} 个文件）")
        except Exception as exc:  # noqa: BLE001 - 恢复失败只警告：删掉复制了一半的残留，该项之后重新生成
            shutil.rmtree(dst, ignore_errors=True)
            print(f"警告：从 PERSIST_DIR 恢复 {name} 失败（{type(exc).__name__}: {exc}），将重新生成")


def sync_to_persist(label: str) -> None:
    """把 WORK_DIR 的内容复制到 PERSIST_DIR（只复制有变化的文件）；失败只警告、不中断。"""
    persist = _persist_dir()
    if persist is None:
        return
    try:
        copied = [f"{src.name}（复制 {n}/{total} 个有变化的文件）"
                  for src in sorted(WORK_DIR.iterdir()) for n, total in [_copy_files(src, persist / src.name)]
                  ] if WORK_DIR.is_dir() else []
        stale = _stale_docs(persist)
        print(f"[{label}] 已同步到 {persist}：" + ("、".join(copied) or "（WORK_DIR 里还没有内容）")
              + ("\n" if stale else ""))
        if stale:
            print(f"提示：PERSIST_DIR/store/docs 里有 {len(stale)} 个本地已不存在的旧文档目录（{'、'.join(stale[:5])}"
                  f"{'…' if len(stale) > 5 else ''}），未删除；集群重启恢复后可能出现同名文档的新旧两份。"
                  "如确认 PERSIST_DIR 不被其他工作目录共用，可手动删除这些目录。")
    except Exception as exc:  # noqa: BLE001
        print(f"警告：同步到 PERSIST_DIR 失败（{type(exc).__name__}: {exc}）；本地结果仍在 {WORK_DIR}")


restore_from_persist()

In [ ]:
# ───────────── 可选：公司自封装 Python SDK 适配器（默认关闭）─────────────
# 公司模型不是 OpenAI 协议、只能通过公司 Python SDK 调用时，把 USE_CUSTOM_LLM 改 True，
# 在 company_chat 里调用公司 SDK，并把模型名配成 company/<任意名>（上面"模型配置"的方式四；
# 问答用 SUPERINDEX_CHAT_MODEL，INDEX_SUMMARY=True 时建库摘要用 SUPERINDEX_INDEX_MODEL，同样写 company/...）。
# 模板把 company_chat 的返回值转成 litellm 需要的格式；流式是"假流式"（整段结果一次性吐出）。
# USE_CUSTOM_LLM=False 时本 cell 什么都不做（不 import litellm）。
USE_CUSTOM_LLM = _env("USE_CUSTOM_LLM", False, _bool)


def company_chat(messages: list[dict], tools: list[dict] | None) -> dict:
    """★ 在这里调用公司 SDK ★

    入参：
      messages  OpenAI 风格的消息列表：{"role": "system"/"user"/"assistant"/"tool", "content": str, ...}；
                assistant 消息可能带 "tool_calls"，tool 消息带 "tool_call_id"（图片等非文字内容已被去掉）。
      tools     OpenAI 风格的工具定义列表 [{"type": "function", "function": {"name", "description", "parameters"}}]，
                可能为 None。问答 agent 靠它调用 search_pages / get_page_content 等，模型必须能返回工具调用。
    返回：{"text": str | None, "tool_calls": [{"id": str, "name": str, "arguments": str(JSON)}]}
      要调用工具时 tool_calls 非空（text 可为 None）；给最终回答时 tool_calls 为空、text 是回答。
    """
    raise NotImplementedError("请在 company_chat 里调用公司 SDK")


def register_company_llm() -> None:
    """把 company_chat 注册成 litellm 的自定义 provider "company"。"""
    import litellm
    from litellm import CustomLLM
    from litellm.types.utils import GenericStreamingChunk

    def text_of(content: Any) -> str:
        if isinstance(content, list):  # 多段内容：只保留文字
            return "".join(p.get("text", "") for p in content if isinstance(p, dict) and p.get("type") == "text")
        return content if isinstance(content, str) else "" if content is None else json.dumps(content, ensure_ascii=False)

    def call(messages: list[dict], optional_params: dict) -> tuple[str | None, list[dict]]:
        result = company_chat([{**m, "content": text_of(m.get("content"))} for m in messages],
                              optional_params.get("tools"))
        calls = [{"id": c["id"], "type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
                 for c in result.get("tool_calls") or []]
        return result.get("text"), calls

    def to_response(result: tuple[str | None, list[dict]], model_response: Any, model: str) -> Any:
        text, calls = result
        model_response.choices[0].message = litellm.Message(content=text, role="assistant", tool_calls=calls or None)
        model_response.choices[0].finish_reason = "tool_calls" if calls else "stop"
        model_response.model = model
        return model_response

    def to_chunks(result: tuple[str | None, list[dict]]) -> list[GenericStreamingChunk]:
        text, calls = result
        usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}  # SDK 有用量数据可在此填入
        chunks = [GenericStreamingChunk(text=text or "", is_finished=False, finish_reason="", usage=usage,
                                        index=0, tool_use=None)]
        chunks += [GenericStreamingChunk(text="", is_finished=False, finish_reason="", usage=usage, index=0,
                                         tool_use={**call, "index": i}) for i, call in enumerate(calls)]
        chunks[-1]["is_finished"] = True
        chunks[-1]["finish_reason"] = "tool_calls" if calls else "stop"
        return chunks

    class CompanyLLM(CustomLLM):
        def completion(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                       encoding, api_key, logging_obj, optional_params, **kwargs):
            return to_response(call(messages, optional_params), model_response, model)

        async def acompletion(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                              encoding, api_key, logging_obj, optional_params, **kwargs):
            return to_response(call(messages, optional_params), model_response, model)

        def streaming(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                      encoding, api_key, logging_obj, optional_params, **kwargs):
            yield from to_chunks(call(messages, optional_params))

        async def astreaming(self, model, messages, api_base, custom_prompt_dict, model_response, print_verbose,
                             encoding, api_key, logging_obj, optional_params, **kwargs):
            for chunk in to_chunks(call(messages, optional_params)):
                yield chunk

    others = [p for p in (litellm.custom_provider_map or []) if p.get("provider") != "company"]
    litellm.custom_provider_map = [*others, {"provider": "company", "custom_handler": CompanyLLM()}]


if USE_CUSTOM_LLM:
    register_company_llm()
    print("已注册自定义 LLM provider：模型名写 company/<任意名>")

## 步骤 1：PDF → Markdown

逐个 PDF 转换（默认 `pdfspine`；`pdfspine_ingest.ipynb` 已转过的 `.md` 直接跳过），`md/` 下已有同名 `.md` 就跳过（`FORCE_EXTRACT=True` 强制重转）。每个 `.md` 旁写一个 `.meta.json`。

- `text-layer`：`Extractor(env={}).document_text(pdf)` 读文字层并写入 `<!-- page: N -->` 页码标记（`env={}` 保证绝不调用 Azure）。
  `ADD_HEADINGS=True` 时再用 `page_index_flash`（离线版面分析，不调 LLM）得到章节树，在对应页的页码标记之后插入 `#` 标题；
  flash 失败或得不到结构时只给警告并退回无标题版本。
  **所有页文字都为空**的 PDF（扫描件）不写 `.md`、不建库：只写 `meta.json`（`status: "skipped_no_text_layer"`）并登记到 `WORK_DIR/skipped_pdfs.json`，下次运行不再重复抽取；
  部分页为空的 PDF 照常处理，空页数量和页码记在 `meta.json` 里并在汇总表给出警告。单个 PDF 出问题都不影响其它文档。
- **带密码的 PDF**：先试空密码（只设了 owner 密码的 PDF 就这样打开，走原有流程），不行再按 `PDF_PASSWORDS` 顺序逐个试。成功后文字层用带密码的 PyPDF2 reader 直接读，
  补标题（flash 的签名不能传密码）用内存里解密好的 PDF（BytesIO）——整个过程**不生成解密副本文件**，落盘的只有 Markdown 和 `meta.json`（只记 `encrypted` / `password_used`，不记密码）。
  没配密码或候选密码都不对：不写 `.md`、不建库，`meta.json` 标 `status: "skipped_wrong_password"` 并登记到 `skipped_pdfs.json`；**下次运行会重新尝试**（密码配置可能已改对）。
  Markdown 是明文，`WORK_DIR` / `PERSIST_DIR` 的访问权限要按原 PDF 的密级管理。
- 汇总表的 `status`：`converted` 已转换 / `skipped_existing` 已有 .md 跳过 / `skipped_no_text_layer` 无文字层 / `skipped_wrong_password` 密码不正确或未配置 / `failed` 失败。
  指向被跳过 PDF 的题（步骤 3-4）不会去问模型，结果里 `status` 记为对应的 `skipped_*`。
- **AES 加密的 PDF** 需要 `pycryptodome`（`notebook` 依赖组已含）；缺失时该 PDF 标为 `failed` 并给出安装提示，其它文档继续处理。
- `pdfspine`：`superindex.extractors.pdfspine.convert_pdf` 离线转换（表格 → Markdown 表格，按字号出 `#` 标题，不再用 flash 补标题），页码标记、`meta.json`、跳过与登记规则同 `text-layer`；带密码的 PDF 由 pdfspine 在内存里解密（不需要 `pycryptodome`），不落解密副本。
- `azure-di`：直接复用 `extractors.azure_di.extract_corpus`（Azure 输出本身带标题和表格）。

In [ ]:
def list_pdfs(pdf_dir: Path) -> list[Path]:
    if not pdf_dir.is_dir():
        raise FileNotFoundError(
            f"PDF 目录不存在：{pdf_dir.resolve()}（来源：{_source('PDF_DIR')}；ROOT_DIR={ROOT_DIR}）。"
            f"修法：在 .env 里写 `PDF_DIR=<PDF 目录的绝对路径>`（.env 放在 {ROOT_DIR} 或 {Path.cwd().resolve()}），"
            "或设置同名环境变量；旧名 SI_NB_PDF_DIR 已不再生效")
    pdfs = sorted(p for p in pdf_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf")
    if not pdfs:
        raise FileNotFoundError(f"{pdf_dir.resolve()} 下没有 PDF 文件")
    return pdfs


def add_headings(md: str, pdf: Path, password: str | None = None) -> tuple[str, int, str | None]:
    """用 flash 离线版面分析给 md 补 # 标题。返回 (新 md, 标题数, 警告)；失败时原样返回 md 并给出警告。
    password 不为 None 时（PDF 需要密码）：flash 不能传密码，改给它一份内存里解密好的 PDF（BytesIO，不落盘）。"""
    try:
        from superindex.engine.flash import page_index_flash

        source = decrypted_stream(pdf, password) if password is not None else str(pdf)
        tree = page_index_flash(source, summary=False, optimize=False)
    except Exception as exc:  # noqa: BLE001 - 失败只退回无标题版本
        return md, 0, f"flash 失败（{type(exc).__name__}: {exc}），已退回无标题版本"

    heads: dict[int, list[str]] = {}

    def walk(nodes: list[dict] | None, level: int) -> None:
        for node in nodes or []:
            title = " ".join(str(node.get("title") or "").split())
            page = node.get("start_index")
            # flash 在找不到结构时会退化成一页一个 "Page N" 节点，那不是真标题
            if title and isinstance(page, int) and not re.fullmatch(r"Page \d+", title):
                heads.setdefault(page, []).append("#" * min(level, 6) + " " + title)
            walk(node.get("nodes"), level + 1)

    walk(tree.get("structure"), 1)
    if not heads:
        return md, 0, f"flash 未得到章节结构（toc_source={tree.get('toc_source')}），已退回无标题版本"

    parts = PAGE_MARKER_RE.split(md)  # [前导文本, 页码1, 页文本1, 页码2, 页文本2, ...]
    out, count = [parts[0]], 0
    for i in range(1, len(parts), 2):
        page, text = int(parts[i]), parts[i + 1]
        out.append(f"<!-- page: {page} -->")
        if page in heads:
            out.append("\n\n" + "\n\n".join(heads[page]) + "\n\n" + text.lstrip("\n"))
            count += len(heads[page])
        else:
            out.append(text)
    return "".join(out), count, None


SKIPPED = "skipped_no_text_layer"      # meta.json 里的 status：无文字层，已跳过
WRONG_PASSWORD = "skipped_wrong_password"  # meta.json 里的 status：需要密码但没配 / 候选密码都不对，已跳过
SKIP_LABELS = {SKIPPED: "无文字层", WRONG_PASSWORD: "密码不正确或未配置"}  # 被跳过的 PDF：状态 -> 原因（读题、汇总共用）
SKIP_STATUSES = tuple(SKIP_LABELS)


def describe_failure(exc: Exception) -> str:
    """失败原因文本；PyPDF2 读 AES 加密 PDF 缺 pycryptodome 时给出明确的中文处理办法。"""
    if "DependencyError" in type(exc).__name__ or "PyCryptodome" in str(exc):
        return ("该 PDF 是 AES 加密的，缺少依赖 pycryptodome；请运行 `uv sync --group notebook`"
                "（或 `uv add pycryptodome`）安装后重跑本 Notebook，其它文档不受影响"
                f"（原始错误：{type(exc).__name__}: {exc}）")
    return f"{type(exc).__name__}: {exc}"


def open_with_password(pdf: Path) -> tuple[bool, str | None, bool]:
    """判断 PDF 怎么打开，返回 (是否加密, 用到的密码, 能否打开)：
    - 不加密：(False, None, True)；加密但空密码即可打开（如只设了 owner 密码）：(True, None, True)——都走原有流程；
    - 需要密码：按 PDF_PASSWORDS 顺序尝试，成功 (True, 该密码, True)；没配或都不对 (True, None, False)。
    PyPDF2 的 decrypt 返回 0 = 密码不对，1 / 2 = 匹配了 user / owner 密码。"""
    import PyPDF2

    with pdf.open("rb") as fh:
        reader = PyPDF2.PdfReader(fh)  # 加密的会先自动试空密码；AES 缺 pycryptodome 在这里抛 DependencyError
        if not reader.is_encrypted:
            return False, None, True
        if int(reader.decrypt("")):
            return True, None, True
        for password in PDF_PASSWORDS:
            if int(reader.decrypt(password)):
                return True, password, True
    return True, None, False


def password_document_text(pdf: Path, password: str) -> str:
    """需要密码的 PDF：用密码打开 PyPDF2 reader 逐页抽文字层（只在内存里解密，不落解密副本）。
    输出格式与 Extractor(env={}).document_text 的文字层路径一致：每页前 `<!-- page: N -->`，页间空一行。"""
    import PyPDF2
    from superindex.extractors.azure_di import PAGE_MARKER

    with pdf.open("rb") as fh:
        reader = PyPDF2.PdfReader(fh)
        reader.decrypt(password)
        pages = [re.sub("[\ud800-\udfff]", "\ufffd", page.extract_text() or "") for page in reader.pages]  # 与引擎一样去掉孤立代理项
    parts: list[str] = []
    for number, text in enumerate(pages, start=1):
        parts += [PAGE_MARKER.format(n=number), text]
    return "\n\n".join(parts)


def decrypted_stream(pdf: Path, password: str) -> io.BytesIO:
    """给 flash 用的解密版 PDF：整份留在内存（BytesIO），不写任何文件，用完随对象释放。
    page_index_flash 底层的 pypdfium2 能传密码，但 flash 的签名没有这个参数，所以改传解密后的内容；
    writer.append 会连书签一起带过去（flash 会用到内嵌目录）。"""
    import PyPDF2

    with pdf.open("rb") as fh:
        reader = PyPDF2.PdfReader(fh)
        reader.decrypt(password)
        writer = PyPDF2.PdfWriter()
        writer.append(reader)
        buf = io.BytesIO()
        writer.write(buf)
    buf.seek(0)
    return buf


def extract_pdfspine(pdf: Path, md_path: Path) -> dict[str, Any]:
    """一个 PDF 用 pdfspine 离线转 Markdown（带页码标记、表格、# 标题），写 .md 与 .meta.json；返回汇总表的一行。
    需要密码的 PDF 由 pdfspine 用 PDF_PASSWORDS 在内存里解密，不落解密副本；跳过规则与 extract_text_layer 相同。"""
    from superindex.extractors.pdfspine import convert_pdf, markdown_stats

    res = convert_pdf(pdf, PDF_PASSWORDS)
    if res.status == WRONG_PASSWORD:
        reason = ("PDF 需要密码才能打开，但没有配置 PDF_INGEST_PASSWORD / PDF_PASSWORDS" if not PDF_PASSWORDS
                  else f"PDF 需要密码才能打开，已尝试的 {len(PDF_PASSWORDS)} 个候选密码均不正确")
        write_meta(pdf, status=WRONG_PASSWORD, encrypted=True, password_used=False, pages=None, reason=reason)
        return {"status": WRONG_PASSWORD, "pages": None, "headings": 0,
                "note": f"警告：{reason}，已跳过、不建库；改好 PDF_INGEST_PASSWORD / PDF_PASSWORDS 后重跑会自动重试"}
    if res.status == SKIPPED:
        write_meta(pdf, status=SKIPPED, encrypted=res.encrypted, password_used=res.password_used, pages=res.pages,
                   empty_pages=res.empty_pages, reason="PDF 没有文字层（扫描件/纯图片），pdfspine 抽取结果全为空；如需处理请改用 azure-di")
        return {"status": SKIPPED, "pages": res.pages, "headings": 0,
                "note": "警告：无文字层（扫描件？），已跳过、不建库；请改用 PDF_EXTRACTOR=\"azure-di\" 处理"}
    stats = markdown_stats(res.markdown)
    md_path.parent.mkdir(parents=True, exist_ok=True)
    md_path.write_text(res.markdown, encoding="utf-8")
    write_meta(pdf, status="ok", encrypted=res.encrypted, password_used=res.password_used, pages=res.pages,
               empty_pages=res.empty_pages, add_headings=False, headings_added=bool(stats["headings"]),
               heading_count=stats["headings"], tables=stats["tables"])
    note = f"警告：{len(res.empty_pages)} 页无文字 {res.empty_pages}（可能是整页图片）" if res.empty_pages else ""
    return {"status": "converted", "pages": res.pages, "headings": stats["headings"], "note": note}


def meta_path_of(pdf: Path) -> Path:
    return (MD_DIR / pdf.relative_to(PDF_DIR)).with_suffix(".meta.json")


def write_meta(pdf: Path, **fields: Any) -> None:
    meta = {"source": str(pdf.resolve()), "source_path": pdf.relative_to(PDF_DIR).as_posix(),
            "extractor": PDF_EXTRACTOR, **fields,
            "extracted_at": datetime.now().isoformat(timespec="seconds")}
    path = meta_path_of(pdf)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")


def read_meta(pdf: Path) -> dict[str, Any]:
    try:
        return json.loads(meta_path_of(pdf).read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return {}


def extract_text_layer(pdf: Path, md_path: Path) -> dict[str, Any]:
    """一个 PDF 走文字层转 Markdown，写 .md 与 .meta.json；返回汇总表的一行。"""
    encrypted, password, openable = open_with_password(pdf)
    if not openable:  # 需要密码但没配 / 候选密码都不对：跳过并标记（reason 里不含任何密码）
        reason = ("PDF 需要密码才能打开，但没有配置 PDF_INGEST_PASSWORD / PDF_PASSWORDS" if not PDF_PASSWORDS
                  else f"PDF 需要密码才能打开，已尝试的 {len(PDF_PASSWORDS)} 个候选密码均不正确")
        write_meta(pdf, status=WRONG_PASSWORD, encrypted=True, password_used=False, pages=None, reason=reason)
        return {"status": WRONG_PASSWORD, "pages": None, "headings": 0,
                "note": f"警告：{reason}，已跳过、不建库；改好 PDF_INGEST_PASSWORD / PDF_PASSWORDS 后重跑会自动重试"}
    used = password is not None  # meta 只记"有没有用密码"，不记是哪一个
    # env={} -> 即使环境里有 Azure 密钥也绝不调用 Azure；需要密码的走内存解密版
    md = password_document_text(pdf, password) if used else Extractor(env={}).document_text(pdf)
    parts = PAGE_MARKER_RE.split(md)
    pages = [(int(parts[i]), parts[i + 1]) for i in range(1, len(parts), 2)]
    empty_pages = [n for n, text in pages if not text.strip()]
    if not pages or len(empty_pages) == len(pages):  # 所有页文字都为空 -> 无文字层，跳过并标记
        write_meta(pdf, status=SKIPPED, encrypted=encrypted, password_used=used, pages=len(pages), empty_pages=empty_pages,
                   reason="PDF 没有文字层（扫描件/纯图片），文字层抽取结果全为空；如需处理请改用 azure-di")
        return {"status": SKIPPED, "pages": len(pages), "headings": 0,
                "note": "警告：无文字层（扫描件？），已跳过、不建库；请改用 PDF_EXTRACTOR=\"azure-di\" 处理"}
    notes = [f"警告：{len(empty_pages)} 页无文字 {empty_pages}（可能是整页图片）"] if empty_pages else []
    headings = 0
    if ADD_HEADINGS:
        md, headings, warn = add_headings(md, pdf, password)
        if warn:
            notes.append("警告：" + warn)
    md_path.parent.mkdir(parents=True, exist_ok=True)
    md_path.write_text(md, encoding="utf-8")
    write_meta(pdf, status="ok", encrypted=encrypted, password_used=used, pages=len(pages), empty_pages=empty_pages,
               add_headings=ADD_HEADINGS, headings_added=bool(headings), heading_count=headings)
    return {"status": "converted", "pages": len(pages), "headings": headings, "note": "；".join(notes)}


print(f"MD_DIR {MD_DIR} 下已有 {sum(1 for _ in MD_DIR.rglob('*.md')) if MD_DIR.is_dir() else 0} 个 .md", flush=True)
pdfs = list_pdfs(PDF_DIR)
# 库里按文档名区分文档：不同文件夹的同名 PDF 建库时自动在文档名前加文件夹前缀区分（步骤 2），先醒目提示
_name_clashes = same_name_sources([p.relative_to(PDF_DIR).as_posix() for p in pdfs])
if _name_clashes:
    _prefixed = clash_names([p.relative_to(PDF_DIR).as_posix() for p in pdfs])
    print("!" * 80 + f"\n注意：{len(_name_clashes)} 组不同文件夹的同名 PDF，建库时已自动加文件夹前缀区分"
          "（只改这些文档的库内名称，其它文档名称不变）：", flush=True)
    for _name, _paths in _name_clashes.items():
        print(f"  {_name}: " + " | ".join(f"{_p} → {_prefixed[_p]}" for _p in _paths), flush=True)
    print("!" * 80, flush=True)
rows: list[dict[str, Any]] = []
step_started = time.perf_counter()
print(f"步骤 1：共 {len(pdfs)} 个 PDF，抽取方式 {PDF_EXTRACTOR}，补标题 {'开' if ADD_HEADINGS and PDF_EXTRACTOR == 'text-layer' else '关'}", flush=True)


def progress(i: int, pdf: Path, row: dict[str, Any], seconds: float) -> None:
    """实时打印一个 PDF 的处理结果（开始处理时已先打印了 "[i/n] 文件名 ..."，这里接着写在同一行）。"""
    pages = f"，{row['pages']} 页" if row.get("pages") else ""
    heads = f"，标题 {row['headings']}" if row.get("headings") else ""
    print(f" {row['status']}{pages}{heads}（{seconds:.1f}s，累计 {time.perf_counter() - step_started:.0f}s）", flush=True)


if PDF_EXTRACTOR in ("text-layer", "pdfspine"):
    for i, pdf in enumerate(pdfs, start=1):
        print(f"[{i}/{len(pdfs)}] {pdf.relative_to(PDF_DIR)} ...", end="", flush=True)
        started = time.perf_counter()
        md_path = (MD_DIR / pdf.relative_to(PDF_DIR)).with_suffix(".md")
        meta = read_meta(pdf)
        # 已标记 skipped_wrong_password 的不在这里跳过：没有 .md、状态也不是 SKIPPED，每次运行都会重新尝试打开（密码配置可能已改对）
        if not FORCE_EXTRACT and (md_path.exists() or meta.get("status") == SKIPPED):
            # 已有 .md，或已被标记为无文字层：不重复抽取（FORCE_EXTRACT=True 可强制重试）
            status = SKIPPED if meta.get("status") == SKIPPED and not md_path.exists() else "skipped_existing"
            note = ""
            if status == SKIPPED:
                note = "此前已标记为无文字层，不重复抽取；改用 azure-di 或设 FORCE_EXTRACT=True 重试"
            elif meta and (meta.get("extractor") != PDF_EXTRACTOR
                           or (PDF_EXTRACTOR == "text-layer" and meta.get("add_headings") != ADD_HEADINGS)):
                note = "已有 .md 的抽取方式/补标题与当前配置不同；需要按新配置重转请设 FORCE_EXTRACT=True"
            if meta.get("empty_pages") and status != SKIPPED:
                note = (note + "；" if note else "") + f"警告：{len(meta['empty_pages'])} 页无文字 {meta['empty_pages']}"
            if meta and not meta.get("source_path"):  # 旧 meta.json：补记来源相对路径（不重转）
                meta_path_of(pdf).write_text(json.dumps({**meta, "source_path": pdf.relative_to(PDF_DIR).as_posix()},
                                                        ensure_ascii=False, indent=2), encoding="utf-8")
            rows.append({"pdf": pdf.name, "status": status, "pages": meta.get("pages"),
                         "headings": meta.get("heading_count"), "note": note})
            progress(i, pdf, rows[-1], time.perf_counter() - started)
            continue
        if FORCE_EXTRACT and md_path.exists():
            md_path.unlink()  # 重转：避免上一次的 .md 在新结果为"无文字层"时残留
        try:
            result = (extract_pdfspine if PDF_EXTRACTOR == "pdfspine" else extract_text_layer)(pdf, md_path)
        except Exception as exc:  # noqa: BLE001 - 单个 PDF 失败不中断其它文档
            result = {"status": "failed", "pages": None, "headings": None,
                      "note": describe_failure(exc)}
        rows.append({"pdf": pdf.name, **result})
        progress(i, pdf, rows[-1], time.perf_counter() - started)
elif PDF_EXTRACTOR == "azure-di":
    # 只有显式选了 azure-di 才会调用 Azure（需 AZURE_DI_ENDPOINT / AZURE_DI_KEY）
    from superindex.extractors.azure_di import AzureDIConfig, AzureDIError, extract_corpus

    try:
        AzureDIConfig.from_env().validate()  # 缺 AZURE_DI_ENDPOINT / AZURE_DI_KEY 时给出明确提示
    except AzureDIError as exc:
        raise RuntimeError(f"PDF_EXTRACTOR=\"azure-di\" 但 Azure 未配置：{exc}\n"
                           "或者把 PDF_EXTRACTOR 改回 \"text-layer\"（离线）。") from exc
    statuses: dict[str, str] = {}

    def on_progress(i: int, n: int, pdf: Path, status: str, info: Any) -> None:
        statuses[str(pdf)] = status
        print(f"[{i}/{n}] {Path(pdf).name} {status}（累计 {time.perf_counter() - step_started:.0f}s）", flush=True)

    summary = extract_corpus(PDF_DIR, MD_DIR, force=FORCE_EXTRACT, workers=2, page_markers=True, on_progress=on_progress)
    errors = dict(e.split(": ", 1) for e in summary["errors"])
    for pdf in pdfs:
        status = {"converted": "converted", "skipped": "skipped_existing"}.get(statuses.get(str(pdf)), "failed")
        rows.append({"pdf": pdf.name, "status": status, "pages": read_meta(pdf).get("pages"), "headings": None,
                     "note": errors.get(pdf.name, "")})
else:
    raise ValueError(f"PDF_EXTRACTOR 只能是 \"text-layer\"、\"pdfspine\" 或 \"azure-di\"，当前是 {PDF_EXTRACTOR!r}")

# 汇总所有被跳过的 PDF（含此前运行标记的），写到 WORK_DIR/skipped_pdfs.json
skipped_pdfs = [{"pdf": str(p.relative_to(PDF_DIR)), "source": str(p.resolve()), "status": read_meta(p)["status"],
                 "reason": read_meta(p).get("reason"), "pages": read_meta(p).get("pages"),
                 "marked_at": read_meta(p).get("extracted_at")}
                for p in pdfs if read_meta(p).get("status") in SKIP_STATUSES]
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "skipped_pdfs.json").write_text(json.dumps(skipped_pdfs, ensure_ascii=False, indent=2), encoding="utf-8")

_counts: dict[str, int] = {}
for row in rows:
    _counts[row["status"]] = _counts.get(row["status"], 0) + 1
print(f"步骤 1 完成：{len(rows)} 个 PDF，用时 {time.perf_counter() - step_started:.0f}s；"
      + "，".join(f"{k} {v}" for k, v in sorted(_counts.items())), flush=True)
display(pd.DataFrame(rows))
for row in rows:
    if row["status"] in (*SKIP_STATUSES, "failed") or "警告" in row["note"]:
        print(f"[{row['status']}] {row['pdf']}: {row['note']}")
print(f"被跳过的 PDF：{len(skipped_pdfs)} 个，清单见 {WORK_DIR / 'skipped_pdfs.json'}")

## 步骤 2：建库

对 `md/` 下每个 Markdown 调 `index_markdown`（`pdf=` 记录源 PDF 关联）。内容没变会自动跳过；
`INDEX_SUMMARY=False` 时不调 LLM。

每个文档记下来源 PDF 相对 `PDF_DIR` 的路径（`doc_meta.source_path`；旧 md 从 `.meta.json` 或按文件名回 `PDF_DIR` 查找），
报告类型按「来源文件夹 > 文件名缩写/全称 > LLM」定（映射见 `config/routing_policy.yaml` 的 `report_types`），
报告期按文件名规范化（`FY2022` / `1H2022` / `2022Q3` / `2022M03` …，文件名能解析就以文件名为准）。
已建库的文档只按规则补全这些字段（不调 LLM、不重建树）；最后打印一行统计：各报告类型文档数、有期间的文档数、来源文件夹无法识别类型的文档数。

In [ ]:
settings = LLMSettings.resolve()
summary_model = None
if INDEX_SUMMARY:
    summary_model = settings.require("index")  # 没配 SUPERINDEX_INDEX_MODEL 会给出明确提示
    configure_litellm()
doc_meta_model = (settings.index_model or settings.chat_model) if DOC_META else None
if doc_meta_model:
    configure_litellm()

md_files = sorted(MD_DIR.rglob("*.md")) if MD_DIR.is_dir() else []
if not md_files:
    raise RuntimeError(f"{MD_DIR} 下没有 Markdown，请先成功运行步骤 1")
pdf_by_stem = page_images.pdf_index(PDF_DIR)
# 不同文件夹的同名 PDF：建库前基于 PDF_DIR 下全部 PDF 一次算好库内文档名（冲突的加文件夹前缀，其余不变），每次运行结果一致；
# 并删掉加前缀之前库里留下的那份无前缀同名文档
_pdf_root = PDF_DIR.resolve()
_all_sources = [p.relative_to(_pdf_root).as_posix() for paths in pdf_by_stem.values() for p in paths]
DOC_NAMES = clash_names(_all_sources)
for _gone in drop_clash_leftovers(STORE, _all_sources):
    print(f"已删除库里加前缀前的无前缀同名文档：{_gone}", flush=True)

index_rows: list[dict[str, Any]] = []
# index_markdown 内部会 asyncio.run(...)，而 Notebook 的主线程已有运行中的事件循环，直接调用会报
# "asyncio.run() cannot be called from a running event loop"；放到单独的工作线程里执行（该线程没有事件循环）。
index_pool = ThreadPoolExecutor(max_workers=1)


def pdf_to_link(md_path: Path) -> Path | None:
    """index_markdown(pdf=...) 的源 PDF 关联，只为页面截图服务（默认关闭）。
    步骤 1 用密码打开的 PDF 不关联：页面截图用的 pypdfium2 没有密码就打不开，传 pdf= 只会得到一条误导性警告
    （"PDF ... not linked"）——即需要密码的 PDF 不支持页面截图，文字问答不受影响。"""
    try:
        if json.loads(md_path.with_suffix(".meta.json").read_text(encoding="utf-8")).get("password_used"):
            return None
    except (OSError, ValueError):
        pass
    return page_images.find_pdf(md_path, pdf_by_stem)


rate_limit = RateLimitPolicy(INDEX_RATE_LIMIT_WAIT, INDEX_RATE_LIMIT_MAX_WAITS)  # 额度耗尽后，后续文档不再等待
index_started = time.perf_counter()
print(f"步骤 2：共 {len(md_files)} 个文档，摘要 {'开（调用模型）' if INDEX_SUMMARY else '关'}", flush=True)
for i, md_path in enumerate(md_files, start=1):
    print(f"[{i}/{len(md_files)}] {md_path.relative_to(MD_DIR)} ...", end="", flush=True)
    started = time.perf_counter()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            source = pdf_source_path(md_path, PDF_DIR, MD_DIR, pdf_by_stem)
        res = index_pool.submit(index_markdown, md_path, STORE, summary_model=summary_model,
                                    backend=settings.index_backend(), force=False,
                                    pdf=pdf_to_link(md_path), doc_meta=DOC_META,
                                    doc_meta_model=doc_meta_model, rate_limit=rate_limit,
                                    source_path=source, name=DOC_NAMES.get(source or "")).result()
        doc_meta = res.doc_meta or {}
        period = " ".join(str(doc_meta[k]) for k in ("period", "report_type") if doc_meta.get(k))
        if not res.skipped:
            status = "已建库"
        elif res.doc_meta_added:
            status = "跳过（未变），补元数据 " + (f"{period}（{doc_meta.get('source')}）" if doc_meta else "")
        else:
            status = "跳过（未变）"
        if doc_meta.get("llm_failed"):
            status += "；元数据 LLM 调用失败，已退回文件名规则（下次运行重试）"
        index_rows.append({"doc_id": res.doc_id, "文档": res.name, "页数": res.pages, "节点数": res.nodes,
                           "状态": status,
                           "报告期": " ".join(str(doc_meta[k]) for k in ("period", "report_type") if doc_meta.get(k))
                           + (f"（{doc_meta.get('source')}）" if doc_meta else ""),
                           "页码标记": res.has_markers, "警告": "；".join(res.warnings)})
    except Exception as exc:  # noqa: BLE001 - 单个文档失败不中断其它文档
        index_rows.append({"doc_id": None, "文档": md_path.name, "页数": None, "节点数": None,
                           "状态": f"失败：{type(exc).__name__}: {exc}", "报告期": "", "页码标记": None, "警告": ""})
    print(f" {index_rows[-1]['状态'][:80]}（{time.perf_counter() - started:.1f}s，"
          f"累计 {time.perf_counter() - index_started:.0f}s）", flush=True)
index_pool.shutdown()
if DOC_META:  # 库里其它缺元数据 / 来源路径、或规则已更新的文档（如 md 已不在 MD_DIR 下）也补上；不重建树
    _filled = backfill_doc_meta(STORE, model=doc_meta_model, backend=settings.index_backend(),
                                rate_limit=rate_limit, pdf_dir=PDF_DIR)
    if _filled:
        print(f"文档元数据补齐 {len(_filled)} 个文档", flush=True)
_with_meta = [m for m in DocStore(str(STORE)).list_metas() if (m.get("metadata") or {}).get("doc_meta")]
print(f"文档元数据：{'开' if DOC_META else '关'}（模型 {doc_meta_model or '无，仅文件名规则'}）；"
      f"有元数据 {len(_with_meta)} 个，其中有报告期 "
      f"{sum(1 for m in _with_meta if m['metadata']['doc_meta'].get('period'))} 个", flush=True)
print("文档元数据统计：" + doc_meta_stats(STORE), flush=True)
print(f"步骤 2 完成：{len(index_rows)} 个文档，用时 {time.perf_counter() - index_started:.0f}s", flush=True)
display(pd.DataFrame(index_rows))
sync_to_persist("建库后")

## 步骤 3：读问题

按 `FIELD_MAP` 把数据集规范化成 `batch.Question`（没有 id 时自动编号 `Q001…`；其余字段进 `extra`；映射值支持 `a.b`、`evidence[].file` 这类路径，见配置 cell），
再检查每题的 `doc` 能否在库里匹配到（复用 `batch._scope`，含用 PDF 文件名 stem 匹配）。匹配不到的题仍会运行，在结果里记为错误。汇总里还会给出按 `doc` 范围的题数分布（全库 / 单文档 / 多文档）。

In [ ]:
# results.jsonl 自己占用的字段名；数据集里与之同名的额外字段会被加前缀 src_，避免覆盖结果字段
STANDARD_KEYS = {"id", "question", "doc", "expected", "scope", "answer", "error", "seconds", "llm_turns",
                 "tool_calls", "pages_read", "score", "prefetch", "prefetch_hit", "page_images", "image_count", "status",
                 "rate_limit_retries"}


def load_records(path: Path, records_key: str | None) -> list[Any]:
    """读 .json（顶层列表，或 {"questions": [...]} / {RECORDS_KEY: [...]}）或 .jsonl。"""
    if not path.is_file():
        raise FileNotFoundError(f"问题文件不存在：{path.resolve()}（请修改配置里的 DATASET_PATH）")
    text = path.read_text(encoding="utf-8-sig")
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix != ".json":
        raise ValueError(f"只支持 .json / .jsonl，当前是 {path.suffix!r}")
    data = json.loads(text)
    if isinstance(data, list):
        return data
    if not isinstance(data, dict):
        raise TypeError(f"{path.name}: 顶层应是列表或对象")
    key = records_key or ("questions" if "questions" in data else None)
    if key is None or not isinstance(data.get(key), list):
        candidates = [k for k, v in data.items() if isinstance(v, list)]
        raise ValueError(f"{path.name}: 顶层是对象，找不到题目列表；请设置 RECORDS_KEY，候选键：{candidates}")
    return data[key]


def _walk(node: Any, segments: list[str]) -> Any:
    """沿路径段取值；取不到返回 None。段以 "[]" 结尾表示展开列表，对每个元素继续取后续路径。"""
    if not segments:
        return node
    segment, rest = segments[0], segments[1:]
    spread = segment.endswith("[]")
    key = segment[:-2] if spread else segment
    if key:
        if not isinstance(node, dict) or key not in node:
            return None
        node = node[key]
    if not spread:
        return _walk(node, rest)
    if not isinstance(node, list):
        return None
    out: list[Any] = []
    for element in node:
        value = _walk(element, rest)
        out.extend(value if isinstance(value, list) else [value])
    return out


def resolve_path(item: dict[str, Any], spec: str) -> Any:
    """按 spec 取值：顶层字段名、"a.b" 嵌套路径、"a[].b" 列表展开；取不到返回 None。
    展开得到的列表会去重（保持首次出现顺序）并丢弃空值。"""
    if spec in item:  # 顶层字段名直接命中（含名字里本身带 "." 的字段）
        return item[spec]
    value = _walk(item, spec.split("."))
    if isinstance(value, list) and "[]" in spec:
        seen: dict[str, Any] = {}
        for v in value:
            if v is not None and v != "":
                seen.setdefault(json.dumps(v, ensure_ascii=False, sort_keys=True), v)
        value = list(seen.values())
    return value


def scalar_of(value: Any) -> Any:
    """id / question / expected 取到列表：空 -> 缺省，1 个元素取该元素，多个用换行拼接。"""
    if isinstance(value, list):
        return None if not value else value[0] if len(value) == 1 else "\n".join(str(v) for v in value)
    return value


def to_questions(items: list[Any], field_map: dict[str, str | None]) -> list[Question]:
    """按字段映射规范化；字段名为 None 或数据里没有该字段（含路径取不到）都当作缺省。"""
    names = {k: field_map.get(k) for k in ("id", "question", "expected", "doc", "pages")}
    # 只有"纯顶层字段名"的映射才把源字段从 extra 里拿走；路径映射（如 evidence[].file）保留整个源字段
    mapped = {v for v in names.values() if v and not re.search(r"[.\[\]]", v)}
    questions: list[Question] = []
    for index, item in enumerate(items, start=1):
        if isinstance(item, str) and names["question"]:
            item = {names["question"]: item}
        if not isinstance(item, dict):
            raise TypeError(f"第 {index} 条：应是对象，实际是 {item!r}")
        values = {k: resolve_path(item, v) for k, v in names.items() if v and k != "pages"}
        for key in ("id", "question", "expected"):
            if key in values:
                values[key] = scalar_of(values[key])
        q = _from_item(values, index)  # 复用 batch 的规范化与自动编号（doc 为列表即多文档）
        if q is None:
            print(f"  跳过第 {index} 条：问题为空")
            continue
        q.extra = {(f"src_{k}" if k in STANDARD_KEYS else k): v  # 其余字段原样带进结果
                   for k, v in item.items() if k not in mapped}
        if names["pages"]:  # batch.gold_pages 读 extra["pages"]：列表，元素为 12 或 "12-13"；多文档为页码并集
            pages = resolve_path(item, names["pages"])
            pages = pages if isinstance(pages, list) else [] if pages in (None, "") else [pages]
            if pages:
                q.extra["pages"] = pages
        questions.append(q)
    seen: set[str] = set()
    for q in questions:
        if q.id in seen:
            raise ValueError(f"问题 id 重复：{q.id!r}")
        seen.add(q.id)
    return questions


items = load_records(DATASET_PATH, RECORDS_KEY)
questions = to_questions(items, FIELD_MAP)
if not questions:
    first = items[0] if items else None
    raise ValueError(f"没有读到任何问题；请检查 FIELD_MAP[\"question\"]。数据第一条：{first!r}")
if LIMIT:
    questions = questions[:LIMIT]

store_docs = [m for m in DocStore(str(STORE)).list_metas() if m.get("status") == "completed"]
# 被步骤 1 跳过的 PDF（无文字层 / 密码不正确或未配置；见 meta.json、skipped_pdfs.json）
skipped_names = {Path(p["pdf"]).name: p for p in skipped_pdfs}


def skipped_pdf_of(q: Question) -> list[str]:
    """q.doc 的每一项都在库里匹配不到、且至少一项对应被跳过的 PDF 时，返回这些 PDF 的文件名。"""
    hit: list[str] = []
    for want in q.doc:
        try:
            _scope(store_docs, [want])
            return []  # 有项能在库里匹配到：照常提问
        except ConfigError:
            hit += [n for n in skipped_names if Path(want).stem.lower() in Path(n).stem.lower()]
    return hit


skipped_q: dict[str, tuple[str, str]] = {}  # 题 id -> (状态, 跳过原因)（这些题不会去问模型）
unmatched: list[tuple[str, list[str], str]] = []
for q in questions:
    if q.doc:
        if names := skipped_pdf_of(q):
            status = skipped_names[names[0]]["status"]  # 指向多个被跳过的 PDF 且原因不同时，以第一个为准
            names = [n for n in names if skipped_names[n]["status"] == status]
            skipped_q[q.id] = (status, f"跳过：源 PDF {SKIP_LABELS[status]}（{', '.join(dict.fromkeys(names))}）")
            continue
        try:
            _scope(store_docs, q.doc)
        except ConfigError as exc:
            unmatched.append((q.id, q.doc, str(exc)))

print(f"题数 {len(questions)}（库内文档 {len(store_docs)} 个）；"
      f"{sum(1 for q in questions if q.doc)} 题指定了文档，{sum(1 for q in questions if not q.doc)} 题在全库范围提问")
print(f"doc 范围分布：全库 {sum(1 for q in questions if not q.doc)} 题 / "
      f"单文档 {sum(1 for q in questions if len(q.doc) == 1)} 题 / "
      f"多文档 {sum(1 for q in questions if len(q.doc) > 1)} 题；"
      f"带 pages 的 {sum(1 for q in questions if q.extra.get('pages'))} 题（按页码判定检索命中）")
for status, label in SKIP_LABELS.items():
    group = {qid: why for qid, (st, why) in skipped_q.items() if st == status}
    if group:
        print(f"以下 {len(group)} 题指向{label}的 PDF，不会去问模型，直接在结果里记为 {status}：")
        for qid, why in group.items():
            print(f"  {qid}: {why}")
if unmatched:
    print(f"以下 {len(unmatched)} 题的 doc 在库里匹配不到（运行时会记为错误，请检查文档名或 PDF 是否已转换）：")
    for qid, doc, why in unmatched:
        print(f"  {qid}: {doc}")
display(pd.DataFrame([{"id": q.id, "question": q.question, "expected": q.expected,
                       "doc": "; ".join(q.doc), **q.extra} for q in questions]).head(10))

## 步骤 4：逐题回答

流程与 `superindex batch`（`cmd_batch`）对齐：同一个 `make_client`（含 `search_pages` / `calculate` 工具）、关键词检索前置（prefetch）、`run_question`，
每题 agent 最多 `MAX_TURNS` 回合（默认 25；≤0 或 `none` 表示不传、沿用 SDK 默认的 10；用 `LLM_EXTRA_HEADERS` 同样的方式包在 `client.chat` 外注入）；
记录字段与 `results.jsonl` 完全兼容，每完成一题立即追加写入；单题失败（含超时）只记在该题的 `error` 里，不中断整批。
`RESUME=True` 时续跑最近一次运行目录：跳过已成功的题，重跑出错的题。
模型接口被限流（错误里有 429 / rate limit / quota 等）时所有线程一起暂停 `RATE_LIMIT_WAIT` 秒（默认 60）再重试该题，最多 `RATE_LIMIT_RETRIES` 次（默认 5，0 = 关闭）；结果只写最终一次，重试次数记在 `rate_limit_retries`。末尾按成功 / 跳过 / 出错原因（回合上限、限流、超时等）分类统计。

In [ ]:
RUNS_DIR.mkdir(parents=True, exist_ok=True)
settings = LLMSettings.resolve()
instructions = resolve_instructions()

existing_runs = sorted(p for p in RUNS_DIR.glob("*") if (p / RESULTS_FILE).is_file())
run_dir = existing_runs[-1] if RESUME and existing_runs else RUNS_DIR / datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir.mkdir(parents=True, exist_ok=True)
results_path = run_dir / RESULTS_FILE
results_path.touch(exist_ok=True)

done = {k: r for k, r in read_results(run_dir).items() if not r.get("error")} if RESUME else {}
todo = [q for q in questions if q.id not in done]
prefetch_k = prefetch.resolve_k(PREFETCH, PREFETCH_K)
image_mode = page_images.resolve_mode(None)
route_policy = _routing_policy() if ROUTE else None
started = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"运行目录 : {run_dir}")
print(f"共 {len(questions)} 题：已完成 {len(questions) - len(todo)}，本次运行 {len(todo)}；"
      f"timeout {TIMEOUT:g}s，max_turns {MAX_TURNS or '默认(10)'}，并发 {CONCURRENCY}，prefetch {prefetch_k or '关'}，page image {image_mode}，"
      f"期间路由 {('开（含下一年）' if ROUTE_ADJACENT else '开（不含下一年）') if ROUTE else '关'}")

def with_extra_headers(client: Any) -> Any:
    """让 client.chat 的每次调用都带上 LLM_EXTRA_HEADERS（run_question / image_chat.chat 不透传 extra_headers）。
    每次调用时才读取 LLM_EXTRA_HEADERS，中途改了字典（如刷新 token）下一题就生效。只覆盖问答路径。"""
    chat = client.chat

    def chat_with_headers(*args: Any, **kwargs: Any) -> Any:
        kwargs["extra_headers"] = {**LLM_EXTRA_HEADERS, **(kwargs.get("extra_headers") or {})}
        return chat(*args, **kwargs)

    client.chat = chat_with_headers
    return client


def with_max_turns(client: Any, max_turns: int | None) -> Any:
    """调用方没传 max_turns 时补上（run_question / image_chat.chat 不传，会落到 SDK 默认 10）；max_turns 为 None 不改动。"""
    if max_turns is None:
        return client
    chat = client.chat

    def chat_with_max_turns(*args: Any, **kwargs: Any) -> Any:
        kwargs.setdefault("max_turns", max_turns)
        return chat(*args, **kwargs)

    client.chat = chat_with_max_turns
    return client


client = None
if any(q.id not in skipped_q for q in todo):  # 只剩被跳过的题时不需要模型
    try:
        client = make_client(settings, STORE, instructions=instructions)  # 没配 SUPERINDEX_CHAT_MODEL 会在这里报错
        if LLM_EXTRA_HEADERS:
            client = with_extra_headers(client)
        client = with_max_turns(client, MAX_TURNS)
    except ConfigError as exc:
        raise RuntimeError(f"未配置可用的问答模型：{exc}\n"
                           "请设置 SUPERINDEX_CHAT_MODEL，或设置 OPENAI_MODEL / OPENAI_BASE_URL / OPENAI_API_KEY"
                           "（.env 或「模型配置」cell 的 LLM_CONFIG）") from exc

lock = threading.Lock()
page_texts: dict[str, list[str]] = {}
bar = tqdm(total=len(todo), desc="回答问题", unit="题") if tqdm and todo else None


def say(text: str) -> None:
    (bar.write if bar else print)(text)


class _SkippedQuestion(Exception):
    """指向被跳过 PDF（无文字层 / 密码不正确或未配置）的题：不问模型。"""


pause_until = 0.0  # 全局限流闸门：此时刻之前所有线程都不发起新的提问（受 lock 保护）


def wait_gate() -> None:
    while True:
        with lock:
            delay = pause_until - time.time()
        if delay <= 0:
            return
        time.sleep(delay)


def close_gate() -> None:
    global pause_until
    with lock:
        pause_until = max(pause_until, time.time() + RATE_LIMIT_WAIT)  # 取 max，不叠加


def ask_once(q: Question) -> dict[str, Any]:
    """问一次（每次都重新检索、新建 page image session）；返回记录，不抛异常。"""
    try:
        if q.id in skipped_q:  # 指向被跳过的 PDF：不问模型，只记一条说明
            raise _SkippedQuestion(skipped_q[q.id][1])
        route = route_scope(store_docs, q.question, q.doc, enabled=ROUTE, adjacent=ROUTE_ADJACENT,
                            policy=route_policy)  # 有 doc：同 _scope；没有：按期间元数据路由
        scope_ids = route.scope_ids
        scope = scope_ids[0] if scope_ids and len(scope_ids) == 1 else scope_ids
        hits = prefetch.search(STORE, q.question, scope_ids, prefetch_k)
        session = page_images.new_session(STORE, image_mode)
        if session is not None:
            session.attach_prefetch(hits)
        wait_gate()  # prefetch / 建 session 期间别的线程可能刚被限流，真正调模型前再看一次闸门
        record = run_question(client, q, scope, timeout=TIMEOUT, reasoning_effort=settings.reasoning_effort,
                              message=prefetch.augment(q.question, hits), session=session)
        if prefetch_k:
            judge, flags = judge_hits(STORE, q, hits, page_texts)
            record["prefetch"] = [{"doc_name": h.doc_name, "page": h.page, "relevant": f}
                                  for h, f in zip(hits, flags)]
            record["prefetch_hit"] = any(flags) if judge else None
        record.update(route.fields())
        record.update(route_diagnostics(record, route, store_docs))
    except Exception as exc:  # noqa: BLE001 - 如 doc 匹配不到：记下来，整批继续
        skipped = isinstance(exc, _SkippedQuestion)
        record = {"id": q.id, "question": q.question, "doc": q.doc, "expected": q.expected, **q.extra,
                  "scope": None, "answer": "", "error": str(exc) if skipped else f"{type(exc).__name__}: {exc}",
                  "seconds": 0.0, "llm_turns": 0, "tool_calls": [], "pages_read": [],
                  "score": score(q.expected, "")}
        if skipped:
            record["status"] = skipped_q[q.id][0]
    return record


def answer_one(q: Question) -> None:
    retries = 0
    while True:
        wait_gate()
        record = ask_once(q)
        limited = is_rate_limited(record["error"])
        if limited and RATE_LIMIT_RETRIES > 0:
            close_gate()  # 重试已用完也关闸，让其它线程同样避开这一分钟
        if not limited or retries >= RATE_LIMIT_RETRIES:
            break
        retries += 1
        say(f"{q.id}  被限流，等待 {RATE_LIMIT_WAIT:g}s 后重试（第 {retries}/{RATE_LIMIT_RETRIES} 次）：{record['error'][:120]}")
    record["rate_limit_retries"] = retries
    with lock:  # 每题完成立即追加写入（只写最终一次）
        with results_path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(record, ensure_ascii=False) + "\n")
        if bar:
            bar.update(1)
        status = (record["error"] if record.get("status") in SKIP_STATUSES else "ERROR " + record["error"]) if record["error"] else \
            ("命中" if (record["score"] or {}).get("hit") else ("未命中" if record["score"] else "已回答"))
        say(f"{q.id}  {record['seconds']:.1f}s  {status}  读取页: {', '.join(record['pages_read']) or '-'}"
            + (f"  路由: {record['route_note']}" if record.get("route_reason") not in (None, "doc") else ""))


t0 = time.time()
if todo:
    with ThreadPoolExecutor(max_workers=max(1, CONCURRENCY)) as pool:
        list(pool.map(answer_one, todo))
if bar:
    bar.close()
wall = time.time() - t0
print(f"本次运行 {len(todo)} 题，用时 {wall:.1f}s")


def _ids(rs: list[dict[str, Any]]) -> str:
    ids = [str(r["id"]) for r in rs]
    return ", ".join(ids[:20]) + (f" 等 {len(ids)} 题" if len(ids) > 20 else "")


run_latest = read_results(run_dir)  # 本运行目录里每题最后一条
run_records = [run_latest[q.id] for q in questions if q.id in run_latest]
ok_records = [r for r in run_records if not r.get("error")]
skip_n = sum(1 for r in run_records if r.get("status") in SKIP_STATUSES)
by_kind: dict[str, list[dict[str, Any]]] = {k: [] for k in ERROR_KINDS}
for r in run_records:
    if r.get("error") and r.get("status") not in SKIP_STATUSES:
        by_kind[error_kind(r["error"])].append(r)
n_hit = sum(1 for r in ok_records if (r.get("score") or {}).get("hit"))
n_noexp = sum(1 for r in ok_records if not r.get("score"))
print(f"结果分类：成功 {len(ok_records)}（命中 {n_hit} / 未命中 {len(ok_records) - n_hit - n_noexp} / 无期望 {n_noexp}），"
      f"跳过 {skip_n}，出错 {sum(len(v) for v in by_kind.values())}")
print("出错原因：" + "；".join(f"{label} {len(by_kind[k])} 题" for k, label in ERROR_KINDS.items()))
if mt := by_kind["max_turns"]:
    print(f"回合上限的题：llm_turns 中位数 {statistics.median(r.get('llm_turns') or 0 for r in mt):g}、"
          f"工具调用次数中位数 {statistics.median(len(r.get('tool_calls') or []) for r in mt):g}；"
          f"其中检索线索含答案页 {sum(1 for r in mt if r.get('prefetch_hit') is True)} 题 / "
          f"不含 {sum(1 for r in mt if r.get('prefetch_hit') is False)} 题 / "
          f"无法判定 {sum(1 for r in mt if r.get('prefetch_hit') is None)} 题")
for k, label in ERROR_KINDS.items():
    if by_kind[k]:
        print(f"  {label}：{_ids(by_kind[k])}")
for line in route_summary_lines(run_records):  # 只有计数，不含题目 / 文档内容
    print(line.removeprefix("- "))
for line in calc_summary_lines(run_records):
    print(line.removeprefix("- "))

## 步骤 4 诊断（可选，可单独运行）

只需先运行「环境」和「配置」两格即可单独运行；只读 `results.jsonl`、不调模型。输出约一屏，把它拍照发回即可（同时另存 `<运行目录>/diagnosis.txt`）。


In [ ]:
# ───────────── 步骤 4 诊断：只读 results.jsonl，不调模型；输出紧凑，方便拍照 ─────────────
# 只依赖「环境」「配置」两格（RUNS_DIR、json、Path、statistics、Counter 等）；不依赖步骤 1~4 是否执行过。
def diagnose_step4() -> None:
    import unicodedata
    from collections import Counter

    W = 110  # 每行最多 110 个显示宽度（中文按 2 计）

    def width(s: str) -> int:
        return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)

    def fit(s: str, limit: int = W) -> str:
        if width(s) <= limit:
            return s
        out, w = "", 0
        for c in s:
            cw = width(c)
            if w + cw > limit - 1:
                break
            out, w = out + c, w + cw
        return out + "…"

    def cut(v: Any, n: int) -> str:
        s = " ".join(str(v if v is not None else "?").split())
        return s if len(s) <= n else s[:n] + "…"

    def wrap(prefix: str, items: list[str], sep: str = "  ", max_lines: int = 99) -> list[str]:
        lines, cur, fresh = [], prefix, True
        for it in items:
            if not fresh and width(cur + sep + it) > W:
                lines.append(cur)
                cur, fresh = "  ", True
            cur += it if fresh else sep + it
            fresh = False
        lines.append(cur)
        if len(lines) > max_lines:
            lines = lines[:max_lines]
            lines[-1] = fit(lines[-1], W - 1) + "…"
        return lines

    def mmm(vals: list[float], fmt: str = "g") -> str:
        return "-" if not vals else f"{min(vals):{fmt}}/{statistics.median(vals):{fmt}}/{max(vals):{fmt}}"

    def calls_of(r: dict[str, Any]) -> list[dict[str, Any]]:
        return [c for c in (r.get("tool_calls") or []) if isinstance(c, dict)]

    def args_of(c: dict[str, Any]) -> dict[str, Any]:
        a = c.get("arguments")
        return a if isinstance(a, dict) else {}

    def pages_of(c: dict[str, Any]) -> list[tuple[str, int]]:
        a, out = args_of(c), []
        for part in str(a.get("pages") or "").replace(" ", "").split(","):
            lo, _, hi = part.partition("-")
            if lo.isdigit():
                out += [(str(a.get("doc_name") or "?"), n) for n in range(int(lo), min(int(hi) if hi.isdigit() else int(lo), int(lo) + 199) + 1)]
        return out

    def step_text(c: dict[str, Any]) -> str:
        name, a = str(c.get("name") or "?"), args_of(c)
        if name == "search_pages":
            inner = f'"{cut(a.get("query"), 30)}"'
        elif name == "get_page_content":
            inner = f"{cut(a.get('doc_name'), 12)}, {cut(a.get('pages'), 12)}"
        elif name == "get_document_structure":
            inner = cut(a.get("doc_name"), 12) + (f", part={a['part']}" if a.get("part") else "")
        else:
            inner = cut(next(iter(a.values()), ""), 20) if a else ""
        return cut(f"{name}({inner})", 50)

    # ── 找到 results.jsonl ──
    fname = globals().get("RESULTS_FILE", "results.jsonl")
    env_path = os.environ.get("RESULTS_PATH", "").strip()
    if env_path:
        p = under_root(env_path)
        path = p / fname if p.is_dir() else p
    elif isinstance(globals().get("run_dir"), Path):
        path = globals()["run_dir"] / fname
    else:
        runs = sorted(d for d in RUNS_DIR.glob("*") if (d / fname).is_file()) if RUNS_DIR.is_dir() else []
        path = runs[-1] / fname if runs else None
    if path is None or not path.is_file():
        print(f"没找到 results.jsonl（{RUNS_DIR} 下没有含 {fname} 的运行目录）；先运行步骤 4，或设环境变量 RESULTS_PATH 指向它。")
        return
    out_dir = path.parent

    latest: dict[str, dict[str, Any]] = {}  # 同一 id 多行取最后一行
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        try:
            rec = json.loads(line)
        except ValueError:
            continue
        if isinstance(rec, dict):
            latest[str(rec.get("id"))] = rec
    recs = list(latest.values())

    skipped = [r for r in recs if str(r.get("status") or "").startswith("skipped")]
    errs = [r for r in recs if r.get("error") and r not in skipped]
    oks = [r for r in recs if not r.get("error")]
    by_kind: dict[str, list[dict[str, Any]]] = {k: [] for k in ERROR_KINDS}
    for r in errs:
        by_kind[error_kind(str(r["error"]))].append(r)
    n_hit = sum(1 for r in oks if (r.get("score") or {}).get("hit"))
    n_noexp = sum(1 for r in oks if not r.get("score"))
    out = [f"诊断 run: {out_dir.resolve()}",
           f"题数 {len(recs)} | 成功 {len(oks)}（命中 {n_hit} / 未命中 {len(oks) - n_hit - n_noexp} / 无期望 {n_noexp}）"
           f"| 跳过 {len(skipped)} | 出错 {len(errs)} | 有过限流重试 {sum(1 for r in recs if (r.get('rate_limit_retries') or 0) > 0)} 题",
           "出错分类: " + " | ".join(f"{lab} {len(by_kind[k])}" for k, lab in
                                    (("max_turns", "回合上限"), ("rate_limit", "限流"), ("timeout", "超时"),
                                     ("backend", "接口其它"), ("other", "其它")))]
    out.append("出错文本 Top5（前 70 字 × 次数）:")
    top = Counter(cut(r["error"], 70) for r in errs).most_common(5)
    out += [fit(f"  {n} × {t}") for t, n in top] or ["  （无）"]

    def prefetch_line(rs: list[dict[str, Any]]) -> str:
        c = Counter(r.get("prefetch_hit") for r in rs)
        return f"含答案页 {c[True]} / 不含 {c[False]} / 无法判定 {len(rs) - c[True] - c[False]}"

    mt = by_kind["max_turns"]
    out.append(f"── 回合上限的题（{len(mt)} 题）──")
    if mt:
        ncalls = [len(calls_of(r)) for r in mt]
        pg = [len({x for c in calls_of(r) if c.get("name") == "get_page_content" for x in pages_of(c)}) for r in mt]
        out.append(fit(f"llm_turns: {mmm([r.get('llm_turns') or 0 for r in mt])}   工具调用数: {mmm(ncalls)}   "
                       f"耗时s: {mmm([float(r.get('seconds') or 0) for r in mt], '.0f')}"))
        out.append(f"读取页数（去重）: {mmm(pg)}   其中 0 页的 {sum(1 for n in pg if n == 0)} 题")
        total, qs = Counter(), Counter()
        for r in mt:
            names = [str(c.get("name") or "?") for c in calls_of(r)]
            total.update(names)
            qs.update(set(names))
        out += wrap("工具使用（总次数/涉及题数）: ", [f"{n} {t}/{qs[n]}" for n, t in total.most_common(6)], ", ", 2) \
            if total else ["工具使用: （无工具调用）"]
        out.append(f"检索线索 prefetch_hit: {prefetch_line(mt)}")

        def ndoc(r: dict[str, Any]) -> int:
            d = r.get("doc")
            return len(d) if isinstance(d, list) else (1 if d else 0)

        sc = Counter("全库" if "scope" in r and r["scope"] is None else "未记录" if "scope" not in r
                     else "单文档" if isinstance(r["scope"], str) else "多文档" for r in mt)
        out.append(fit(f"题集 doc 字段: 指定 1 个文档 {sum(1 for r in mt if ndoc(r) == 1)} 题 / 多个 {sum(1 for r in mt if ndoc(r) > 1)} 题"
                       f" / 未指定 {sum(1 for r in mt if ndoc(r) == 0)} 题    scope: 单文档 {sc['单文档']} / 多文档 {sc['多文档']}"
                       f" / 全库 {sc['全库']}" + (f" / 未记录 {sc['未记录']}" if sc["未记录"] else "")))

        def max_run(r: dict[str, Any]) -> int:
            best = cur = 0
            for c in calls_of(r):
                cur = cur + 1 if c.get("name") == "search_pages" else 0
                best = max(best, cur)
            return best

        def dup_q(r: dict[str, Any]) -> bool:
            q = Counter(str(args_of(c).get("query")).strip().casefold() for c in calls_of(r)
                        if c.get("name") == "search_pages" and args_of(c).get("query"))
            return any(n >= 2 for n in q.values())

        def dup_page(r: dict[str, Any]) -> bool:
            seen: Counter = Counter()
            for c in calls_of(r):
                if c.get("name") == "get_page_content":
                    seen.update(set(pages_of(c)))
            return any(n >= 2 for n in seen.values())

        out.append(fit(f"重复行为: 同一 search 关键词重复≥2 次的 {sum(dup_q(r) for r in mt)} 题；同一页重复读取≥2 次的 "
                       f"{sum(dup_page(r) for r in mt)} 题；连续≥5 次都是 search 的 {sum(1 for r in mt if max_run(r) >= 5)} 题"))
        last = Counter(str(calls_of(r)[-1].get("name") or "?") if calls_of(r) else "(无调用)" for r in mt)
        out += wrap("最后一步工具: ", [f"{k} {n} 题" for k, n in last.most_common(5)], " / ", 2)
    if oks:
        out.append(f"── 对照：成功的题（{len(oks)} 题）──")
        c = Counter(r.get("prefetch_hit") for r in oks)
        out.append(fit(f"llm_turns: {mmm([r.get('llm_turns') or 0 for r in oks])}   工具调用数: {mmm([len(calls_of(r)) for r in oks])}   "
                       f"耗时s: {mmm([float(r.get('seconds') or 0) for r in oks], '.0f')}   prefetch_hit: 含 {c[True]} / 不含 {c[False]}"))
    if mt:
        ordered = sorted(mt, key=lambda r: (len(calls_of(r)), str(r.get("id"))))
        picks = list(dict.fromkeys(id(ordered[i]) for i in (len(ordered) - 1, len(ordered) // 2, 0)))
        out.append(f"── 回合上限样例 {len(picks)} 题（工具调用序列，按调用顺序）──")
        for r in (x for x in ordered if id(x) in picks):
            out.append(fit(f"{r.get('id')} | {cut(r.get('question'), 40)} | 共 {len(calls_of(r))} 步"))
            cs = calls_of(r)[:12]
            out += wrap("  ", [f"{i} {step_text(c)}" for i, c in enumerate(cs, 1)], "  ", 4) if cs else ["  （无工具调用）"]
    others = [(k, lab, by_kind[k][0]) for k, lab in ERROR_KINDS.items() if k != "max_turns" and by_kind[k]]
    if others:
        out.append("── 其它出错类别各 1 例 ──")
        out += [fit(f"{r.get('id')} | {lab} | {cut(r['error'], 90)}") for _, lab, r in others]
    ids = [str(r.get("id")) for r in errs]
    if ids:
        shown = ids[:60]
        out += wrap(f"出错题 id（{len(ids)} 题）: ", shown + ([f"…等 {len(ids)} 题"] if len(ids) > 60 else []), ", ", 3)
    else:
        out.append("出错题 id: 无")

    text = "\n".join(out)
    print(text)
    diag_path = out_dir / "diagnosis.txt"
    try:
        diag_path.write_text(text + "\n", encoding="utf-8")
        print(f"已另存: {diag_path.resolve()}")
    except OSError as exc:
        print(f"（diagnosis.txt 写入失败：{exc}）")


diagnose_step4()


## 步骤 5：整理保存

`read_results` + `write_summary` 生成 `summary.md`；再用 pandas 导出 `answers.csv`（utf-8-sig，Excel 直接打开不乱码）和 `answers.xlsx`，只有三列：`question`、`expected`、`answer`（出错 / 跳过的题 `answer` 为空，`expected` 缺失为空）。

In [ ]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

latest = read_results(run_dir)
records = [latest[q.id] for q in questions if q.id in latest]
write_summary(records, run_dir / SUMMARY_FILE, {
    "started": started, "questions": str(DATASET_PATH), "store": str(STORE),
    "chat_model": settings.chat_model, "wall_seconds": wall, "prefetch_k": prefetch_k, "page_image": image_mode})

def to_row(r: dict[str, Any]) -> dict[str, Any]:
    return {"question": r.get("question"), "expected": r.get("expected") or "",
            "answer": "" if r.get("error") else r.get("answer") or ""}


df = pd.DataFrame([to_row(r) for r in records], columns=["question", "expected", "answer"])

csv_path, xlsx_path = run_dir / "answers.csv", run_dir / "answers.xlsx"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
def clean_cell(v: Any) -> Any:
    return ILLEGAL_CHARACTERS_RE.sub("", v)[:32767] if isinstance(v, str) else v  # xlsx 不接受控制字符，单元格最长 32767 字符


# DataFrame.map 需要 pandas>=2.1；更旧的版本退回 applymap（这样不必升级集群预装的 pandas）
(df.map(clean_cell) if hasattr(df, "map") else df.applymap(clean_cell)).to_excel(xlsx_path, index=False)

skip_counts = {s: sum(1 for r in records if r.get("status") == s) for s in SKIP_STATUSES}
skipped = sum(skip_counts.values())
errors = sum(1 for r in records if r.get("error")) - skipped
scored = [r for r in records if r.get("score") and not r.get("error")]
hits = sum(1 for r in scored if r["score"]["hit"])
print(f"题数 {len(records)}：成功 {len(records) - errors - skipped}，失败 {errors}，跳过 {skipped}"
      + (f"（{'，'.join(f'{SKIP_LABELS[s]} {n}' for s, n in skip_counts.items() if n)}）" if skipped else "")
      + (f"；粗评分命中 {hits}/{len(scored)}（{hits / len(scored):.0%}，仅供筛查，需人工复核）" if scored else ""))
print("产出文件：")
for path in (run_dir / RESULTS_FILE, run_dir / SUMMARY_FILE, csv_path, xlsx_path):
    print(f"  {path}")
display(df)
sync_to_persist("全部完成")

## 步骤 6：逐题打印答案

结果已落盘后，把 `records`（步骤 5 读出的完整记录，按题集原顺序，含续跑前已成功的题）里每个问题的 id 和答案一个一个打印出来；答案不截断，出错的题打印 status 和 error。

In [ ]:
total = len(records)
for n, r in enumerate(records, start=1):
    print("=" * 80, flush=True)
    print(f"[{n}/{total}] id: {r.get('id')}", flush=True)
    print(f"问题: {r.get('question')}", flush=True)
    if r.get("error"):
        print(f"状态: {r.get('status') or 'error'}\n错误: {r['error']}", flush=True)
    print(f"答案:\n{r.get('answer') or '（无答案）'}", flush=True)
print("=" * 80, flush=True)